# Classificacao por Quadrantes Emocionais - Fingerprints por Quadrante

Este notebook cria quatro classificadores binarios independentes: Q1 vs resto, Q2 vs resto, Q3 vs resto e Q4 vs resto.

A metrica principal de selecao e analise e `positive_f1_mean`, o F1 da classe positiva do quadrante alvo na validacao `GroupKFold` por `song_id`.


## 1. Imports


In [1]:
import os
import re
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from tqdm import tqdm
from IPython.display import display, Markdown

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from scipy.stats import ttest_rel, wilcoxon

try:
    from sklearn.model_selection import StratifiedGroupKFold
except ImportError:
    StratifiedGroupKFold = None

print(f"pandas {pd.__version__}  |  numpy {np.__version__}")



pandas 2.1.4  |  numpy 1.26.4


## 2. Configuracao


In [2]:
# ===========================
# CAMINHOS
# ===========================
PROJECT_DIR = Path(r"C:\dev\python\TCC Ajustado")
DATA_DIR    = PROJECT_DIR / "Dados"

FINGERPRINT_BLOCK_PATH = DATA_DIR / "fingerprint_band_rank" / "fingerprint_band_rank.parquet"
FINGERPRINT_RANK_BLOCK_PATH = DATA_DIR / "fingerprint_rank" / "fingerprint_rank.parquet"

# Baseline openSMILE: resultados já salvos pelo notebook de quadrantes openSMILE
BASELINE_TABLES_PATH = DATA_DIR / "pycaret_quadrants_outputs" / "tables"

OUTPUT_PATH   = DATA_DIR / "pycaret_quadrants_fingerprints_by_quadrant_outputs"
FIGURES_PATH  = OUTPUT_PATH / "figures"
TABLES_PATH   = OUTPUT_PATH / "tables"
MODELS_PATH   = OUTPUT_PATH / "models"
for p in [FIGURES_PATH, TABLES_PATH, MODELS_PATH]:
    p.mkdir(parents=True, exist_ok=True)

# ===========================
# COLUNAS
# ===========================
SONG_ID_COL   = "song_id"
BLOCK_ID_COL  = "block_id"
START_COL     = "block_start_sec"
END_COL       = "block_end_sec"
VALENCE_COL   = "valence"
AROUSAL_COL   = "arousal"
QUADRANT_COL  = "quadrant_label_std"   # coluna re-derivada neste notebook

META_COLS = {
    SONG_ID_COL, BLOCK_ID_COL, START_COL, END_COL,
    "block_duration_sec", "valence", "arousal",
    "quadrant", "quadrant_label", "quadrant_label_std",
    "method", "title", "artist", "genre", "filename",
    # band-rank raw-peaks metadata columns
    "band_name", "band_id", "band_low_hz", "band_high_hz", "topk_per_band",
    # legacy / fallback names
    "block_idx", "start_time", "end_time", "duration_sec",
}

# ===========================
# QUADRANTES — mesmo protocolo do baseline openSMILE
# ===========================
# Escala VA: –1 a 1  →  limiar = 0
VALENCE_THRESHOLD = 0.0
AROUSAL_THRESHOLD = 0.0
VA_SCALE = "-1_to_1"

QUADRANT_ORDER = [
    "Q1_Alegre_Energetico",
    "Q2_Tenso_Raivoso",
    "Q3_Triste_Melancolico",
    "Q4_Calmo_Relaxado",
]
QUADRANT_COLOR_MAP = {
    "Q1_Alegre_Energetico":  "#2E7D32",
    "Q2_Tenso_Raivoso":      "#C62828",
    "Q3_Triste_Melancolico": "#1565C0",
    "Q4_Calmo_Relaxado":     "#F9A825",
}
QUADRANT_SHORT = {
    "Q1_Alegre_Energetico":  "Q1 Alegre/Energ.",
    "Q2_Tenso_Raivoso":      "Q2 Tenso/Raivoso",
    "Q3_Triste_Melancolico": "Q3 Triste/Melanc.",
    "Q4_Calmo_Relaxado":     "Q4 Calmo/Relaxado",
}

# ===========================
# MODELAGEM
# ===========================
RANDOM_STATE      = 42
N_SPLITS          = 5
SELECTOR_K_VALUES = [30, 60, 100, 200]
MAX_MISSING_RATE  = 0.40
MAX_SAMPLE_FOR_EDA = 50_000

# ===========================
# GRÁFICOS
# ===========================
EXPORT_HTML     = True
EXPORT_PNG      = False
EXPORT_SVG      = False
FIG_WIDTH       = 1200
FIG_HEIGHT      = 720
FIG_SCALE       = 2
PLOT_TEMPLATE   = "plotly_white"
SHOW_FIGURES    = True

print("Saída:", OUTPUT_PATH)
print("Protocolo: 4 classificadores binarios one-vs-rest por quadrante")



## 3. Funcoes utilitarias


In [3]:
def save_table(df_out, filename):
    path = TABLES_PATH / filename
    df_out.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Tabela salva: {path}")
    return path


def save_fig(fig, name, width=None, height=None, scale=None, show=None):
    layout_kwargs = {"template": PLOT_TEMPLATE}
    if width is None and fig.layout.width is None:
        width = FIG_WIDTH
    if height is None and fig.layout.height is None:
        height = FIG_HEIGHT
    if width is not None:
        layout_kwargs["width"] = width
    if height is not None:
        layout_kwargs["height"] = height
    fig.update_layout(**layout_kwargs)

    show_fig = SHOW_FIGURES if show is None else bool(show)
    if show_fig:
        fig.show()

    saved_paths = {}
    export_scale = FIG_SCALE if scale is None else scale
    if EXPORT_HTML:
        try:
            path = FIGURES_PATH / f"{name}.html"
            fig.write_html(str(path), include_plotlyjs="cdn")
            saved_paths["html"] = path
            print(f"Salvo: {path.name}")
        except Exception as e:
            print(f"[AVISO] HTML não exportado: {e}")
    if EXPORT_PNG:
        try:
            path = FIGURES_PATH / f"{name}.png"
            fig.write_image(str(path), scale=export_scale)
            saved_paths["png"] = path
        except Exception as e:
            print(f"[AVISO] PNG não exportado: {e}")
    if EXPORT_SVG:
        try:
            path = FIGURES_PATH / f"{name}.svg"
            fig.write_image(str(path))
            saved_paths["svg"] = path
        except Exception as e:
            print(f"[AVISO] SVG não exportado: {e}")
    return saved_paths



def safe_sample(df_in, max_rows=MAX_SAMPLE_FOR_EDA, random_state=RANDOM_STATE):
    if len(df_in) <= max_rows:
        return df_in.copy()
    return df_in.sample(max_rows, random_state=random_state).copy()


def numeric_feature_cols(df, exclude=None):
    exclude = set(exclude or [])
    result = []
    for col in df.columns:
        if col in exclude:
            continue
        s = pd.to_numeric(df[col], errors="coerce")
        if s.notna().sum() == 0 or s.nunique(dropna=True) <= 1:
            continue
        result.append(col)
    return result


def feature_missing_rate(df, cols):
    return df[cols].isnull().mean()


print("Utilitários carregados.")



## 4. Carregamento e preparacao dos dados


In [4]:
_band_df = pd.read_parquet(FINGERPRINT_BLOCK_PATH)
_band_df.columns = [str(c).strip() for c in _band_df.columns]

_rank_df = pd.read_parquet(FINGERPRINT_RANK_BLOCK_PATH)
_rank_df.columns = [str(c).strip() for c in _rank_df.columns]

# Identify feature columns (non-meta) in each file and prefix them
_band_feat = [c for c in _band_df.columns if c not in META_COLS]
_rank_feat = [c for c in _rank_df.columns if c not in META_COLS]
_band_df = _band_df.rename(columns={c: f"fpband__{c}" for c in _band_feat})
_rank_df = _rank_df.rename(columns={c: f"fprank__{c}" for c in _rank_feat})

_MERGE_KEYS = [SONG_ID_COL, BLOCK_ID_COL, START_COL, END_COL]
_rank_prefixed = [f"fprank__{c}" for c in _rank_feat]
raw_df = _band_df.merge(
    _rank_df[_MERGE_KEYS + _rank_prefixed],
    on=_MERGE_KEYS,
    how="inner",
)

print(f"Band Rank: {_band_df.shape[0]:,} blocos, {len(_band_feat)} features")
print(f"Rank: {_rank_df.shape[0]:,} blocos, {len(_rank_feat)} features")
print(f"Dataset merged: {raw_df.shape[0]:,} linhas × {raw_df.shape[1]} colunas")
print("Colunas de metadados presentes:", [c for c in raw_df.columns if c in META_COLS])



Band Rank: 6,506 blocos, 347 features
Rank: 6,506 blocos, 208 features
Dataset merged: 6,506 linhas × 566 colunas
Colunas de metadados presentes: ['song_id', 'filename', 'block_id', 'block_start_sec', 'block_end_sec', 'block_duration_sec', 'valence', 'arousal', 'quadrant', 'quadrant_label', 'method']


In [5]:
def add_standardized_quadrant(df, v_thresh=VALENCE_THRESHOLD, a_thresh=AROUSAL_THRESHOLD):
    """Re-deriva quadrant_label_std com os mesmos limiares do baseline openSMILE."""
    out  = df.copy()
    v    = pd.to_numeric(out[VALENCE_COL], errors="coerce")
    a    = pd.to_numeric(out[AROUSAL_COL], errors="coerce")
    high_v = v >= v_thresh
    high_a = a >= a_thresh

    conditions = [
        high_v & high_a,
        ~high_v & high_a,
        ~high_v & ~high_a,
        high_v & ~high_a,
    ]
    labels = [
        "Q1_Alegre_Energetico",
        "Q2_Tenso_Raivoso",
        "Q3_Triste_Melancolico",
        "Q4_Calmo_Relaxado",
    ]
    out[QUADRANT_COL] = np.select(conditions, labels, default=np.nan)
    # remove linhas sem VA válido
    out = out.dropna(subset=[QUADRANT_COL]).copy()
    return out


df = add_standardized_quadrant(raw_df)

print(f"Após filtro VA: {df.shape[0]:,} blocos")
print("Distribuição dos quadrantes:")
print(df[QUADRANT_COL].value_counts())



Após filtro VA: 6,506 blocos
Distribuição dos quadrantes:
quadrant_label_std
Q1_Alegre_Energetico     3217
Q3_Triste_Melancolico    1382
Q2_Tenso_Raivoso         1019
Q4_Calmo_Relaxado         888
Name: count, dtype: int64


In [6]:
# Identifica colunas de features de fingerprint
all_feature_cols = [c for c in df.columns if c not in META_COLS]
all_feature_cols = [c for c in all_feature_cols if c not in [
    SONG_ID_COL, BLOCK_ID_COL, START_COL, END_COL,
    VALENCE_COL, AROUSAL_COL, "block_duration_sec",
    "quadrant", "quadrant_label", "method", "title", "artist", "genre",
    QUADRANT_COL,
]]

# Remove features com missing excessivo ou sem variância
missing_rate = feature_missing_rate(df, all_feature_cols)
usable_feature_cols = [
    c for c in all_feature_cols
    if missing_rate[c] <= MAX_MISSING_RATE
    and pd.to_numeric(df[c], errors="coerce").nunique(dropna=True) > 1
]

print(f"Features totais: {len(all_feature_cols)}")
print(f"Features utilizáveis (missing ≤ {MAX_MISSING_RATE:.0%}, variância > 0): {len(usable_feature_cols)}")



Features totais: 555
Features utilizáveis (missing ≤ 40%, variância > 0): 509


In [7]:
# Auditoria anti-vazamento: detecta colunas sensiveis que nao podem entrar como feature
leakage_terms = ["valence", "arousal", "quadrant", "song_id", "title", "artist"]
leakage_audit_rows = []
usable_feature_set = set(usable_feature_cols)
for column in df.columns:
    column_lower = str(column).lower()
    matched_terms = [term for term in leakage_terms if term in column_lower]
    leakage_audit_rows.append({
        "column": column,
        "in_meta_cols": column in META_COLS,
        "in_all_feature_cols": column in all_feature_cols,
        "in_usable_feature_cols": column in usable_feature_set,
        "matched_terms": ",".join(matched_terms),
        "leakage_risk": bool(matched_terms),
        "status": "blocked" if matched_terms else "ok",
    })

leakage_audit_fingerprints = pd.DataFrame(leakage_audit_rows)
save_table(leakage_audit_fingerprints, "leakage_audit_fingerprints.csv")

suspicious_features = leakage_audit_fingerprints[
    leakage_audit_fingerprints["leakage_risk"] & leakage_audit_fingerprints["in_usable_feature_cols"]
].copy()
if not suspicious_features.empty:
    display(Markdown("### Auditoria anti-vazamento - colunas bloqueadas"))
    display(suspicious_features)
    raise RuntimeError(
        "Auditoria anti-vazamento encontrou colunas sensiveis dentro de usable_feature_cols: "
        f"{suspicious_features['column'].tolist()}"
    )

display(Markdown("### Auditoria anti-vazamento - resumo"))
display(leakage_audit_fingerprints["status"].value_counts().to_frame(name="count"))


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\leakage_audit_fingerprints.csv


### Auditoria anti-vazamento - resumo

,count
status,
ok,561
blocked,6


## 5. Inventario e ranking exploratorio das features


In [8]:
def fingerprint_band(feature):
    raw = str(feature).replace("fpband__", "").replace("fprank__", "").lower()
    if "very_high" in raw:
        return "very_high"
    if "_high_" in raw or raw.endswith("_high"):
        return "high"
    if "_mid_" in raw or raw.endswith("_mid"):
        return "mid"
    if "_low_" in raw or raw.endswith("_low"):
        return "low"
    return "global"


def fingerprint_metric(feature):
    raw = str(feature).lower()
    if "freq" in raw:             return "frequency"
    if "mag" in raw:              return "magnitude"
    if "rank" in raw:             return "rank"
    if "midi" in raw or "pitch" in raw or "octave" in raw or "semitone" in raw:
                                  return "pitch_octave"
    if "count" in raw or raw.endswith("_n") or "peak" in raw:
                                  return "counts"
    if "std" in raw:              return "dispersion"
    if "mean" in raw:             return "central_tendency"
    return "other"


def fingerprint_column_technique(feature):
    raw = str(feature).replace("fpband__", "").replace("fprank__", "").lower()
    if raw.startswith("n_"):
        return "global_event_counts"
    if "_top" in raw:
        return "top_k_peak_descriptor"
    if "octdup" in raw or "octave_duplicate" in raw:
        return "octave_duplicate_signal"
    if "unique_pitch" in raw or "event_count" in raw:
        return "pitch_event_counts"
    if raw.endswith("_mean") or raw.endswith("_std"):
        return "band_summary_statistic"
    return "other"


def fingerprint_origin(feature):
    raw = str(feature).lower()
    if raw.startswith("fpband__"):
        return "band_rank"
    if raw.startswith("fprank__"):
        return "rank"
    return "other"


inventory_rows = []
for feat in usable_feature_cols:
    inventory_rows.append({
        "feature": feat,
        "banda": fingerprint_band(feat),
        "metrica": fingerprint_metric(feat),
        "tecnica_coluna": fingerprint_column_technique(feat),
        "origem_feature": fingerprint_origin(feat),
        "missing_rate": float(missing_rate.get(feat, np.nan)),
    })
inventory = pd.DataFrame(inventory_rows)
save_table(inventory, "feature_inventory_fingerprints.csv")

print("DistribuiÃ§Ã£o por banda:")
display(inventory["banda"].value_counts().to_frame())
print("DistribuiÃ§Ã£o por mÃ©trica:")
display(inventory["metrica"].value_counts().to_frame())
print("DistribuiÃ§Ã£o por tÃ©cnica de coluna:")
display(inventory["tecnica_coluna"].value_counts().to_frame())


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\feature_inventory_fingerprints.csv
DistribuiÃ§Ã£o por banda:


,count
banda,
global,213
low,74
mid,74
high,74
very_high,74


DistribuiÃ§Ã£o por mÃ©trica:


,count
metrica,
magnitude,164
pitch_octave,140
rank,108
frequency,85
other,12


DistribuiÃ§Ã£o por tÃ©cnica de coluna:


,count
tecnica_coluna,
top_k_peak_descriptor,420
other,39
band_summary_statistic,38
pitch_event_counts,6
octave_duplicate_signal,3
global_event_counts,3


In [9]:
from sklearn.feature_selection import f_classif

X_anova = df[usable_feature_cols].apply(pd.to_numeric, errors="coerce")
X_anova = X_anova.fillna(X_anova.median())
y_anova = df[QUADRANT_COL].astype(str)

f_scores, _ = f_classif(X_anova, y_anova)

ranking = inventory.copy()
ranking["anova_f"] = f_scores
ranking = ranking.sort_values("anova_f", ascending=False).reset_index(drop=True)
save_table(ranking, "feature_ranking_fingerprints.csv")

display(Markdown("### Top-20 features por ANOVA F-score multiclasse (diagnostico exploratorio)"))
display(ranking[["feature", "banda", "metrica", "anova_f"]].head(20))



Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\feature_ranking_fingerprints.csv


### Top-20 features por ANOVA F-score multiclasse (diagnostico exploratorio)

,feature,banda,metrica,anova_f
0,fpband__energy_db_high,high,other,990.493879
1,fprank__fp_magnitude_mean,global,magnitude,820.255112
2,fprank__fp_magnitude_mean_norm_block,global,magnitude,742.455630
3,fpband__fp_very_high_top_9_magnitude_db,very_high,magnitude,713.461314
4,fpband__fp_high_top_10_magnitude_db,high,magnitude,711.954089
5,fpband__fp_very_high_top_7_magnitude_db,very_high,magnitude,711.851367
6,fpband__fp_very_high_top_8_magnitude_db,very_high,magnitude,710.756098
7,fpband__fp_very_high_top_10_magnitude_db,very_high,magnitude,709.239731
8,fpband__fp_very_high_mag_mean_db,very_high,magnitude,708.201929
9,fpband__fp_very_high_top_5_magnitude_db,very_high,magnitude,706.266428


In [10]:
# Ranking ANOVA por banda
band_anova = (
    ranking.groupby("banda")
    .agg(n_features=("feature", "count"), anova_f_mean=("anova_f", "mean"), anova_f_max=("anova_f", "max"))
    .reset_index()
    .sort_values("anova_f_mean", ascending=False)
)
save_table(band_anova, "band_anova_summary_fingerprints.csv")

fig = px.bar(
    band_anova.sort_values("anova_f_mean"),
    x="anova_f_mean", y="banda", orientation="h",
    color="banda",
    error_x="anova_f_max",
    text=band_anova.sort_values("anova_f_mean")["anova_f_mean"].map(lambda x: f"{x:.1f}"),
    hover_data={"n_features": True, "anova_f_max": ":.1f", "anova_f_mean": ":.2f"},
    title="Poder discriminativo médio (ANOVA F) por banda de fingerprint",
    labels={"anova_f_mean": "ANOVA F médio", "banda": "Banda"},
)
fig.update_traces(textposition="outside")
fig.update_layout(
    showlegend=False,
    margin=dict(l=10, r=80, t=60, b=10),
)
save_fig(fig, "band_anova_fingerprints")



## 6. Helpers de modelagem binaria


In [11]:
# ===========================
# MODELAGEM BINARIA POR QUADRANTE
# ===========================
BINARY_NEGATIVE_LABEL = 0
BINARY_POSITIVE_LABEL = 1
BEST_SORT_COLUMNS = ["positive_f1_mean", "balanced_accuracy_mean", "positive_recall_mean"]


def target_display_name(target_quadrant):
    if "QUADRANT_SHORT" in globals():
        return QUADRANT_SHORT.get(str(target_quadrant), str(target_quadrant))
    if "QUADRANT_SHORT_LABEL" in globals():
        return QUADRANT_SHORT_LABEL.get(str(target_quadrant), str(target_quadrant))
    return str(target_quadrant)


def target_code(target_quadrant):
    return str(target_quadrant).split("_")[0].lower()


def make_binary_target(data_or_labels, target_quadrant):
    """Retorna 1 para o quadrante alvo e 0 para todos os demais."""
    if isinstance(data_or_labels, pd.DataFrame):
        labels = data_or_labels[QUADRANT_COL]
    else:
        labels = pd.Series(data_or_labels, index=getattr(data_or_labels, "index", None))
    return labels.astype(str).eq(str(target_quadrant)).astype(int).rename("target_label")


def build_candidate_models():
    return {
        "Dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
        "Dummy_stratified": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
        "LogisticRegression_balanced": LogisticRegression(
            max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE,
        ),
        "RidgeClassifier_balanced": RidgeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "RandomForest_balanced": RandomForestClassifier(
            n_estimators=250, class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "ExtraTrees_balanced": ExtraTreesClassifier(
            n_estimators=250, class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "KNeighbors": KNeighborsClassifier(n_neighbors=15, weights="distance"),
        "DecisionTree_balanced": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "GaussianNB": GaussianNB(),
    }


def build_pipeline(estimator, selector_k=None, n_features=None):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if selector_k is not None:
        k_effective = min(int(selector_k), int(n_features))
        steps.append(("selector", SelectKBest(score_func=f_classif, k=k_effective)))
    steps.append(("scaler", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)


def selector_k_from_name(selector_name):
    if selector_name is None or str(selector_name) == "none":
        return None
    match = re.search(r"k=(\d+)", str(selector_name))
    return int(match.group(1)) if match else None


def binary_classification_metrics(y_true, y_pred):
    y_true = pd.Series(y_true).astype(int)
    y_pred = pd.Series(y_pred).astype(int)
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "positive_precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "positive_recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "positive_f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "negative_precision": precision_score(y_true, y_pred, pos_label=0, zero_division=0),
        "negative_recall": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "negative_f1": f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        "support_positive": int((y_true == 1).sum()),
        "support_negative": int((y_true == 0).sum()),
    }


def evaluate_binary_groupkfold(pipeline, X, y_binary, groups, cv):
    fold_rows = []
    oof_pred = pd.Series(index=y_binary.index, dtype="float")
    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y_binary, groups), start=1):
        model = clone(pipeline)
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y_binary.iloc[train_idx], y_binary.iloc[test_idx]
        if y_train.nunique() < 2:
            raise RuntimeError(f"Fold {fold_idx} possui apenas uma classe no treino.")
        model.fit(X_train, y_train)
        pred = pd.Series(model.predict(X_test), index=y_test.index).astype(int)
        oof_pred.iloc[test_idx] = pred.values
        row = {
            "fold": fold_idx,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_groups_train": pd.Series(groups.iloc[train_idx]).nunique(),
            "n_groups_test": pd.Series(groups.iloc[test_idx]).nunique(),
        }
        row.update(binary_classification_metrics(y_test, pred))
        fold_rows.append(row)
    return pd.DataFrame(fold_rows), oof_pred.astype(int)


def summarize_binary_fold_metrics(fold_df):
    metric_cols = [
        "accuracy", "balanced_accuracy",
        "positive_precision", "positive_recall", "positive_f1",
        "negative_precision", "negative_recall", "negative_f1",
    ]
    summary = {}
    for metric in metric_cols:
        summary[f"{metric}_mean"] = fold_df[metric].mean()
        summary[f"{metric}_std"] = fold_df[metric].std(ddof=1)
    return summary


def sort_binary_results(results_df):
    if results_df.empty:
        return results_df
    out = results_df.copy()
    out["_target_order"] = out["target_quadrant"].map({q: i for i, q in enumerate(QUADRANT_ORDER)})
    out["_status_ok"] = out["status"].eq("ok")
    sort_cols = ["_target_order", "_status_ok"] + [c for c in BEST_SORT_COLUMNS if c in out.columns]
    ascending = [True, False] + [False] * (len(sort_cols) - 2)
    return out.sort_values(sort_cols, ascending=ascending, na_position="last").drop(columns=["_target_order", "_status_ok"])


def select_best_per_quadrant(results_df, exclude_dummy=True):
    rows = []
    if results_df.empty:
        return pd.DataFrame()
    ok = results_df[results_df["status"].eq("ok")].copy()
    for target_quadrant in QUADRANT_ORDER:
        subset = ok[ok["target_quadrant"].eq(target_quadrant)].copy()
        if subset.empty:
            continue
        candidates = subset
        if exclude_dummy:
            non_dummy = subset[~subset["model"].astype(str).str.startswith("Dummy")].copy()
            if not non_dummy.empty:
                candidates = non_dummy
        rows.append(sort_binary_results(candidates).iloc[0])
    return pd.DataFrame(rows).reset_index(drop=True)


def run_binary_benchmark(X, quadrant_labels, groups, cv, feature_set_name):
    models = build_candidate_models()
    selector_options = [None] + [k for k in SELECTOR_K_VALUES if k < X.shape[1]]
    results_rows = []
    fold_metric_frames = []
    errors = []
    for target_quadrant in tqdm(QUADRANT_ORDER, desc="Quadrantes"):
        y_binary = make_binary_target(quadrant_labels, target_quadrant)
        n_positive = int(y_binary.sum())
        n_negative = int(len(y_binary) - n_positive)
        if n_positive == 0 or n_negative == 0:
            errors.append({"target_quadrant": target_quadrant, "model": "all", "selector": "all", "error": "target sem classes positiva e negativa"})
            continue
        for model_name, estimator in tqdm(models.items(), desc=target_code(target_quadrant), leave=False):
            for selector_k in selector_options:
                selector_name = "none" if selector_k is None else f"SelectKBest(k={selector_k})"
                pipeline = build_pipeline(estimator, selector_k=selector_k, n_features=X.shape[1])
                base_row = {
                    "target_quadrant": target_quadrant,
                    "target_label": target_display_name(target_quadrant),
                    "model": model_name,
                    "selector": selector_name,
                    "n_samples": len(y_binary),
                    "n_positive": n_positive,
                    "n_negative": n_negative,
                    "positive_rate": n_positive / len(y_binary),
                    "n_groups": groups.nunique(),
                    "n_features_requested": X.shape[1],
                    "n_features_effective": X.shape[1] if selector_k is None else min(selector_k, X.shape[1]),
                    "cv": f"GroupKFold({cv.get_n_splits()}) por song_id",
                    "feature_set": feature_set_name,
                }
                try:
                    fold_df, _ = evaluate_binary_groupkfold(pipeline, X, y_binary, groups, cv)
                    fold_df.insert(0, "selector", selector_name)
                    fold_df.insert(0, "model", model_name)
                    fold_df.insert(0, "target_label", target_display_name(target_quadrant))
                    fold_df.insert(0, "target_quadrant", target_quadrant)
                    fold_metric_frames.append(fold_df)
                    row = {**base_row, "status": "ok"}
                    row.update(summarize_binary_fold_metrics(fold_df))
                    results_rows.append(row)
                except Exception as exc:
                    row = {**base_row, "status": f"error: {type(exc).__name__}: {exc}"}
                    results_rows.append(row)
                    errors.append({"target_quadrant": target_quadrant, "model": model_name, "selector": selector_name, "error": str(exc)})
    results = sort_binary_results(pd.DataFrame(results_rows))
    fold_metrics = pd.concat(fold_metric_frames, ignore_index=True) if fold_metric_frames else pd.DataFrame()
    if errors:
        print(f"[AVISO] {len(errors)} erros durante a avaliacao binaria.")
        display(pd.DataFrame(errors).head(20))
    return results, fold_metrics


def generate_best_binary_artifacts(best_rows, X, source_df, quadrant_labels, groups, cv, feature_set_name):
    prediction_frames, report_rows, confusion_rows, fold_frames = [], [], [], []
    fitted_models = {}
    meta_candidates = [
        SONG_ID_COL, "block_id", "block_idx", "block_start_sec", "block_end_sec",
        "start_time", "end_time", VALENCE_COL, AROUSAL_COL, QUADRANT_COL,
    ]
    meta_cols = [c for c in meta_candidates if c in source_df.columns]
    candidates = build_candidate_models()
    for _, best in best_rows.iterrows():
        target_quadrant = best["target_quadrant"]
        model_name = best["model"]
        selector_name = best["selector"]
        selector_k = selector_k_from_name(selector_name)
        y_binary = make_binary_target(quadrant_labels, target_quadrant)
        pipeline = build_pipeline(candidates[model_name], selector_k=selector_k, n_features=X.shape[1])
        fold_df, oof_pred = evaluate_binary_groupkfold(pipeline, X, y_binary, groups, cv)
        fold_df.insert(0, "selector", selector_name)
        fold_df.insert(0, "model", model_name)
        fold_df.insert(0, "target_label", target_display_name(target_quadrant))
        fold_df.insert(0, "target_quadrant", target_quadrant)
        fold_frames.append(fold_df)

        pred_df = source_df[meta_cols].copy()
        pred_df.insert(0, "target_label", target_display_name(target_quadrant))
        pred_df.insert(0, "target_quadrant", target_quadrant)
        pred_df["y_true_binary"] = y_binary.values
        pred_df["prediction"] = oof_pred.astype(int).values
        pred_df["correct"] = pred_df["y_true_binary"].eq(pred_df["prediction"])
        pred_df["model"] = model_name
        pred_df["selector"] = selector_name
        prediction_frames.append(pred_df)

        report = classification_report(
            y_binary,
            oof_pred.astype(int),
            labels=[0, 1],
            target_names=["resto", target_display_name(target_quadrant)],
            output_dict=True,
            zero_division=0,
        )
        report_df = pd.DataFrame(report).T.reset_index().rename(columns={"index": "label"})
        report_df.insert(0, "selector", selector_name)
        report_df.insert(0, "model", model_name)
        report_df.insert(0, "target_label", target_display_name(target_quadrant))
        report_df.insert(0, "target_quadrant", target_quadrant)
        report_rows.append(report_df)

        cm = confusion_matrix(y_binary, oof_pred.astype(int), labels=[0, 1])
        cm_norm = confusion_matrix(y_binary, oof_pred.astype(int), labels=[0, 1], normalize="true")
        label_map = {0: "resto", 1: target_display_name(target_quadrant)}
        for actual_idx, actual_value in enumerate([0, 1]):
            for pred_idx, pred_value in enumerate([0, 1]):
                confusion_rows.append({
                    "target_quadrant": target_quadrant,
                    "target_label": target_display_name(target_quadrant),
                    "model": model_name,
                    "selector": selector_name,
                    "actual": label_map[actual_value],
                    "predicted": label_map[pred_value],
                    "n": int(cm[actual_idx, pred_idx]),
                    "normalized_by_actual": float(cm_norm[actual_idx, pred_idx]) if np.isfinite(cm_norm[actual_idx, pred_idx]) else np.nan,
                })

        final_model = clone(pipeline).fit(X, y_binary)
        fitted_models[target_quadrant] = {
            "model": final_model,
            "features": list(X.columns),
            "target_quadrant": target_quadrant,
            "target_label": target_display_name(target_quadrant),
            "model_name": model_name,
            "selector": selector_name,
            "feature_set": feature_set_name,
        }
    predictions = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
    reports = pd.concat(report_rows, ignore_index=True) if report_rows else pd.DataFrame()
    confusions = pd.DataFrame(confusion_rows)
    best_fold_metrics = pd.concat(fold_frames, ignore_index=True) if fold_frames else pd.DataFrame()
    return predictions, reports, confusions, best_fold_metrics, fitted_models


def run_binary_ablation_study(feature_groups, X_full, quadrant_labels, groups, cv, study_name, feature_set_name, ablation_model_names=None):
    if ablation_model_names is None:
        ablation_model_names = ["LogisticRegression_balanced", "RandomForest_balanced", "ExtraTrees_balanced"]
    candidates = build_candidate_models()
    ablation_models = {name: candidates[name] for name in ablation_model_names if name in candidates}
    rows = []
    for target_quadrant in tqdm(QUADRANT_ORDER, desc=f"Ablation {study_name}"):
        y_binary = make_binary_target(quadrant_labels, target_quadrant)
        n_positive = int(y_binary.sum())
        n_negative = int(len(y_binary) - n_positive)
        if n_positive == 0 or n_negative == 0:
            continue
        for group_label, feat_cols in feature_groups.items():
            feat_cols = [c for c in feat_cols if c in X_full.columns]
            if not feat_cols:
                continue
            X_sub = X_full[feat_cols].copy()
            for model_name, estimator in ablation_models.items():
                pipeline = build_pipeline(estimator, selector_k=None, n_features=len(feat_cols))
                base_row = {
                    "target_quadrant": target_quadrant,
                    "target_label": target_display_name(target_quadrant),
                    "study": study_name,
                    "group": group_label,
                    "model": model_name,
                    "selector": "none",
                    "feature_set": feature_set_name,
                    "n_features": len(feat_cols),
                    "n_samples": len(y_binary),
                    "n_positive": n_positive,
                    "n_negative": n_negative,
                    "positive_rate": n_positive / len(y_binary),
                    "cv": f"GroupKFold({cv.get_n_splits()}) por song_id",
                }
                try:
                    fold_df, _ = evaluate_binary_groupkfold(pipeline, X_sub, y_binary, groups, cv)
                    row = {**base_row, "status": "ok"}
                    row.update(summarize_binary_fold_metrics(fold_df))
                    rows.append(row)
                except Exception as exc:
                    rows.append({**base_row, "status": f"error: {type(exc).__name__}: {exc}"})
    return sort_binary_results(pd.DataFrame(rows))


def best_ablation_per_group(ablation_df):
    if ablation_df.empty:
        return pd.DataFrame()
    ok = ablation_df[ablation_df["status"].eq("ok")].copy()
    if ok.empty:
        return pd.DataFrame()
    sort_cols = ["target_quadrant", "group"] + [c for c in BEST_SORT_COLUMNS if c in ok.columns]
    ascending = [True, True] + [False] * (len(sort_cols) - 2)
    return (
        ok.sort_values(sort_cols, ascending=ascending, na_position="last")
        .drop_duplicates(subset=["target_quadrant", "group"])
        .reset_index(drop=True)
    )


def plot_best_binary_results(best_rows, filename, title):
    if best_rows.empty:
        return
    plot_df = best_rows.copy()
    fig = px.bar(
        plot_df,
        x="target_label",
        y="positive_f1_mean",
        color="target_quadrant",
        error_y="positive_f1_std",
        text=plot_df["positive_f1_mean"].map(lambda x: f"{x:.3f}"),
        hover_data={
            "model": True,
            "selector": True,
            "balanced_accuracy_mean": ":.3f",
            "positive_recall_mean": ":.3f",
            "positive_precision_mean": ":.3f",
            "n_features_effective": True,
            "target_quadrant": False,
        },
        color_discrete_map=QUADRANT_COLOR_MAP,
        title=title,
        labels={
            "positive_f1_mean": "F1 da classe positiva (média GroupKFold)",
            "target_label": "Quadrante alvo",
        },
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(
        showlegend=False,
        yaxis=dict(range=[0, 1.1]),
        margin=dict(l=10, r=10, t=60, b=10),
    )
    save_fig(fig, filename)


def plot_ablation_summary(best_df, filename, title):
    if best_df.empty:
        return
    pivot = best_df.pivot_table(
        index="group", columns="target_label",
        values="positive_f1_mean", aggfunc="max",
    )
    fig = px.imshow(
        pivot,
        text_auto=".3f",
        aspect="auto",
        color_continuous_scale="Blues",
        zmin=0,
        zmax=1,
        title=title,
    )
    fig.update_layout(
        xaxis_title="Quadrante alvo",
        yaxis_title="Grupo de features",
        coloraxis_colorbar_title="F1 positivo",
        margin=dict(l=10, r=10, t=60, b=10),
    )
    save_fig(fig, filename)


def plot_metrics_radar_by_quadrant(best_rows, filename, title):
    """Gráfico de barras agrupado: F1, Precision, Recall e Balanced Acc por quadrante."""
    if best_rows.empty:
        return
    metrics_cfg = [
        ("positive_f1_mean",        "F1 positivo"),
        ("positive_precision_mean",  "Precision"),
        ("positive_recall_mean",     "Recall"),
        ("balanced_accuracy_mean",   "Balanced Acc"),
    ]
    frames = []
    for col, label in metrics_cfg:
        if col not in best_rows.columns:
            continue
        sub = best_rows[["target_label", "target_quadrant", col]].copy()
        sub = sub.rename(columns={col: "valor"})
        sub["metrica"] = label
        frames.append(sub)
    if not frames:
        return
    plot_df = pd.concat(frames, ignore_index=True)
    fig = px.bar(
        plot_df,
        x="target_label",
        y="valor",
        color="target_quadrant",
        facet_col="metrica",
        barmode="group",
        text=plot_df["valor"].map("{:.3f}".format),
        color_discrete_map=QUADRANT_COLOR_MAP,
        title=title,
        labels={"valor": "Valor", "target_label": "Quadrante"},
    )
    fig.update_traces(texttemplate="%{text}", textposition="outside", textfont_size=8, showlegend=False)
    fig.update_layout(
        yaxis=dict(range=[0, 1.15]),
        margin=dict(l=10, r=10, t=80, b=10),
    )
    fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
    fig.update_yaxes(matches=None)
    save_fig(fig, filename)


print("Helpers binarios carregados.")



In [12]:
FEATURE_SET_NAME = "fingerprints"
FEATURE_SET_LABEL = "Fingerprints"
MODEL_SOURCE_DF = df
SAVE_MODEL_ARTIFACTS = True

BINARY_RESULTS_FILENAME = "binary_groupkfold_results_fingerprints.csv"
BINARY_FOLD_METRICS_FILENAME = "binary_groupkfold_fold_metrics_fingerprints.csv"
BINARY_BEST_FILENAME = "binary_groupkfold_best_models_fingerprints.csv"
BINARY_OOF_FILENAME = "binary_oof_predictions_best_models_fingerprints.csv"
BINARY_REPORT_FILENAME = "binary_classification_report_best_models_fingerprints.csv"
BINARY_CONFUSION_FILENAME = "binary_confusion_matrix_best_models_fingerprints.csv"
BINARY_BEST_FOLD_FILENAME = "binary_groupkfold_best_fold_metrics_fingerprints.csv"
BINARY_MODEL_JOBLIB = "best_binary_models_fingerprints.joblib"



In [13]:
def positive_score_from_estimator(model, X_test):
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test)
        proba = np.asarray(proba)
        if proba.ndim == 2 and proba.shape[1] > 1:
            return pd.Series(proba[:, 1], index=X_test.index, dtype=float)
        if proba.ndim == 1:
            return pd.Series(proba, index=X_test.index, dtype=float)
    if hasattr(model, "decision_function"):
        raw_scores = np.asarray(model.decision_function(X_test), dtype=float)
        if raw_scores.ndim > 1:
            raw_scores = raw_scores[:, -1]
        raw_scores = np.clip(raw_scores, -60, 60)
        scaled_scores = 1.0 / (1.0 + np.exp(-raw_scores))
        return pd.Series(scaled_scores, index=X_test.index, dtype=float)
    fallback_scores = pd.Series(model.predict(X_test), index=X_test.index).astype(float)
    return fallback_scores


def evaluate_binary_groupkfold(pipeline, X, y_binary, groups, cv):
    fold_rows = []
    oof_pred = pd.Series(index=y_binary.index, dtype="float")
    oof_score = pd.Series(index=y_binary.index, dtype="float")
    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y_binary, groups), start=1):
        model = clone(pipeline)
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y_binary.iloc[train_idx], y_binary.iloc[test_idx]
        if y_train.nunique() < 2:
            raise RuntimeError(f"Fold {fold_idx} possui apenas uma classe no treino.")
        model.fit(X_train, y_train)
        pred = pd.Series(model.predict(X_test), index=y_test.index).astype(int)
        score = positive_score_from_estimator(model, X_test).astype(float)
        oof_pred.iloc[test_idx] = pred.values
        oof_score.iloc[test_idx] = score.values
        row = {
            "fold": fold_idx,
            "n_train": len(train_idx),
            "n_test": len(test_idx),
            "n_groups_train": pd.Series(groups.iloc[train_idx]).nunique(),
            "n_groups_test": pd.Series(groups.iloc[test_idx]).nunique(),
        }
        row.update(binary_classification_metrics(y_test, pred))
        fold_rows.append(row)
    return pd.DataFrame(fold_rows), oof_pred.astype(int), oof_score.astype(float)


def generate_best_binary_artifacts(best_rows, X, source_df, quadrant_labels, groups, cv, feature_set_name):
    prediction_frames, report_rows, confusion_rows, fold_frames = [], [], [], []
    fitted_models = {}
    meta_candidates = [
        SONG_ID_COL, "block_id", "block_idx", "block_start_sec", "block_end_sec",
        "start_time", "end_time", VALENCE_COL, AROUSAL_COL, QUADRANT_COL,
    ]
    meta_cols = [c for c in meta_candidates if c in source_df.columns]
    candidates = build_candidate_models()
    for _, best in best_rows.iterrows():
        target_quadrant = best["target_quadrant"]
        model_name = best["model"]
        selector_name = best["selector"]
        selector_k = selector_k_from_name(selector_name)
        y_binary = make_binary_target(quadrant_labels, target_quadrant)
        pipeline = build_pipeline(candidates[model_name], selector_k=selector_k, n_features=X.shape[1])
        fold_df, oof_pred, oof_score = evaluate_binary_groupkfold(pipeline, X, y_binary, groups, cv)
        fold_df.insert(0, "selector", selector_name)
        fold_df.insert(0, "model", model_name)
        fold_df.insert(0, "target_label", target_display_name(target_quadrant))
        fold_df.insert(0, "target_quadrant", target_quadrant)
        fold_frames.append(fold_df)

        pred_df = source_df[meta_cols].copy()
        pred_df.insert(0, "target_label", target_display_name(target_quadrant))
        pred_df.insert(0, "target_quadrant", target_quadrant)
        pred_df["y_true_binary"] = y_binary.values
        pred_df["positive_score"] = oof_score.astype(float).values
        pred_df["prediction"] = oof_pred.astype(int).values
        pred_df["prediction_at_0_5"] = (pred_df["positive_score"] >= 0.5).astype(int)
        pred_df["correct"] = pred_df["y_true_binary"].eq(pred_df["prediction"])
        pred_df["model"] = model_name
        pred_df["selector"] = selector_name
        prediction_frames.append(pred_df)

        report = classification_report(
            y_binary,
            oof_pred.astype(int),
            labels=[0, 1],
            target_names=["resto", target_display_name(target_quadrant)],
            output_dict=True,
            zero_division=0,
        )
        report_df = pd.DataFrame(report).T.reset_index().rename(columns={"index": "label"})
        report_df.insert(0, "selector", selector_name)
        report_df.insert(0, "model", model_name)
        report_df.insert(0, "target_label", target_display_name(target_quadrant))
        report_df.insert(0, "target_quadrant", target_quadrant)
        report_rows.append(report_df)

        cm = confusion_matrix(y_binary, oof_pred.astype(int), labels=[0, 1])
        cm_norm = confusion_matrix(y_binary, oof_pred.astype(int), labels=[0, 1], normalize="true")
        label_map = {0: "resto", 1: target_display_name(target_quadrant)}
        for actual_idx, actual_value in enumerate([0, 1]):
            for pred_idx, pred_value in enumerate([0, 1]):
                confusion_rows.append({
                    "target_quadrant": target_quadrant,
                    "target_label": target_display_name(target_quadrant),
                    "model": model_name,
                    "selector": selector_name,
                    "actual": label_map[actual_value],
                    "predicted": label_map[pred_value],
                    "n": int(cm[actual_idx, pred_idx]),
                    "normalized_by_actual": float(cm_norm[actual_idx, pred_idx]) if np.isfinite(cm_norm[actual_idx, pred_idx]) else np.nan,
                })

        final_model = clone(pipeline).fit(X, y_binary)
        fitted_models[target_quadrant] = {
            "model": final_model,
            "features": list(X.columns),
            "target_quadrant": target_quadrant,
            "target_label": target_display_name(target_quadrant),
            "model_name": model_name,
            "selector": selector_name,
            "feature_set": feature_set_name,
        }
    predictions = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
    reports = pd.concat(report_rows, ignore_index=True) if report_rows else pd.DataFrame()
    confusions = pd.DataFrame(confusion_rows)
    best_fold_metrics = pd.concat(fold_frames, ignore_index=True) if fold_frames else pd.DataFrame()
    return predictions, reports, confusions, best_fold_metrics, fitted_models


def build_threshold_analysis(best_predictions, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.10, 0.91, 0.10), 2)
    if best_predictions is None or best_predictions.empty:
        return pd.DataFrame()
    required_cols = {"target_quadrant", "target_label", "y_true_binary", "positive_score"}
    if not required_cols.issubset(set(best_predictions.columns)):
        return pd.DataFrame()
    rows = []
    for _, group_df in best_predictions.groupby(["target_quadrant", "target_label", "model", "selector"], dropna=False):
        y_true = group_df["y_true_binary"].astype(int)
        scores = pd.to_numeric(group_df["positive_score"], errors="coerce")
        valid_mask = y_true.notna() & scores.notna()
        if valid_mask.sum() == 0:
            continue
        y_true = y_true.loc[valid_mask].astype(int)
        scores = scores.loc[valid_mask].astype(float)
        base_row = {
            "target_quadrant": group_df["target_quadrant"].iloc[0],
            "target_label": group_df["target_label"].iloc[0],
            "model": group_df["model"].iloc[0],
            "selector": group_df["selector"].iloc[0],
            "n_samples": int(len(y_true)),
            "n_positive": int((y_true == 1).sum()),
            "n_negative": int((y_true == 0).sum()),
            "positive_rate": float((y_true == 1).mean()),
        }
        for threshold in thresholds:
            y_pred = (scores >= threshold).astype(int)
            metrics = binary_classification_metrics(y_true, y_pred)
            rows.append({
                **base_row,
                "threshold": float(threshold),
                "predicted_positive_rate": float((y_pred == 1).mean()),
                **{
                    "positive_precision_mean": float(metrics["positive_precision"]),
                    "positive_recall_mean": float(metrics["positive_recall"]),
                    "positive_f1_mean": float(metrics["positive_f1"]),
                    "balanced_accuracy_mean": float(metrics["balanced_accuracy"]),
                },
            })
    out = pd.DataFrame(rows)
    if not out.empty:
        out = out.sort_values(["target_quadrant", "threshold"]).reset_index(drop=True)
    return out


def best_ablation_per_quadrant(ablation_df, ablation_type):
    if ablation_df.empty:
        return pd.DataFrame()
    ok = ablation_df[ablation_df["status"].eq("ok")].copy()
    if ok.empty:
        return pd.DataFrame()
    rows = []
    for target_quadrant in QUADRANT_ORDER:
        subset = ok[ok["target_quadrant"].eq(target_quadrant)].copy()
        if subset.empty:
            continue
        best_row = sort_binary_results(subset).iloc[0].to_dict()
        best_row["ablation_type"] = ablation_type
        best_row["feature_group"] = best_row.pop("group", None)
        rows.append(best_row)
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    keep_cols = [
        "target_quadrant", "target_label", "ablation_type", "feature_group", "model",
        "positive_f1_mean", "positive_precision_mean", "positive_recall_mean",
        "balanced_accuracy_mean", "n_features",
    ]
    available = [c for c in keep_cols if c in out.columns]
    return out[available].reset_index(drop=True)


def build_best_fingerprint_ablation_by_quadrant(*named_frames):
    frames = []
    for ablation_type, ablation_df in named_frames:
        best_frame = best_ablation_per_quadrant(ablation_df, ablation_type)
        if not best_frame.empty:
            frames.append(best_frame)
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    return out.sort_values(["target_quadrant", "ablation_type"]).reset_index(drop=True)


def build_fingerprint_vs_opensmile_comparison(best_reports, baseline_tables_path=BASELINE_TABLES_PATH):
    if best_reports is None or best_reports.empty:
        return pd.DataFrame()
    fingerprint_positive = best_reports.loc[
        best_reports["label"].astype(str).eq(best_reports["target_label"].astype(str))
    ].copy()
    fingerprint_positive = fingerprint_positive[[
        "target_quadrant", "target_label", "model", "selector",
        "precision", "recall", "f1-score", "support",
    ]].rename(columns={
        "precision": "fingerprint_precision",
        "recall": "fingerprint_recall",
        "f1-score": "fingerprint_f1",
        "support": "fingerprint_support",
    })
    baseline_report_path = baseline_tables_path / "classification_report_best_model.csv"
    if not baseline_report_path.exists():
        fingerprint_positive["opensmile_model"] = "LogisticRegression_balanced [SelectKBest(k=100)]"
        for column in ["opensmile_precision", "opensmile_recall", "opensmile_f1", "opensmile_support", "f1_delta", "precision_delta", "recall_delta"]:
            fingerprint_positive[column] = np.nan
        return fingerprint_positive
    baseline_report = pd.read_csv(baseline_report_path)
    baseline_positive = baseline_report.loc[
        baseline_report["label"].astype(str).isin(fingerprint_positive["target_label"].astype(str))
    ].copy()
    baseline_positive = baseline_positive.rename(columns={
        "precision": "opensmile_precision",
        "recall": "opensmile_recall",
        "f1-score": "opensmile_f1",
        "support": "opensmile_support",
    })[["label", "opensmile_precision", "opensmile_recall", "opensmile_f1", "opensmile_support"]]
    out = fingerprint_positive.merge(baseline_positive, left_on="target_label", right_on="label", how="left")
    out = out.drop(columns=["label"])
    out["opensmile_model"] = "LogisticRegression_balanced [SelectKBest(k=100)]"
    out["f1_delta"] = out["fingerprint_f1"] - out["opensmile_f1"]
    out["precision_delta"] = out["fingerprint_precision"] - out["opensmile_precision"]
    out["recall_delta"] = out["fingerprint_recall"] - out["opensmile_recall"]
    return out.sort_values("target_quadrant").reset_index(drop=True)


def build_fingerprint_final_summary(best_rows):
    if best_rows is None or best_rows.empty:
        return pd.DataFrame()
    columns = [
        "target_quadrant", "target_label", "model", "selector",
        "positive_f1_mean", "positive_f1_std", "positive_precision_mean", "positive_recall_mean",
        "balanced_accuracy_mean", "n_features_effective", "n_positive", "n_negative",
    ]
    available = [c for c in columns if c in best_rows.columns]
    out = best_rows[available].copy()
    if "n_features_effective" in out.columns:
        out = out.rename(columns={"n_features_effective": "n_features"})
    return out.reset_index(drop=True)


print("Helpers binarios atualizados: score OOF, threshold sweep, comparacao com openSMILE e resumo de ablation.")


Helpers binarios atualizados: score OOF, threshold sweep, comparacao com openSMILE e resumo de ablation.


In [14]:
def run_binary_benchmark(X, quadrant_labels, groups, cv, feature_set_name):
    models = build_candidate_models()
    selector_options = [None] + [k for k in SELECTOR_K_VALUES if k < X.shape[1]]
    results_rows = []
    fold_metric_frames = []
    errors = []
    for target_quadrant in tqdm(QUADRANT_ORDER, desc="Quadrantes"):
        y_binary = make_binary_target(quadrant_labels, target_quadrant)
        n_positive = int(y_binary.sum())
        n_negative = int(len(y_binary) - n_positive)
        if n_positive == 0 or n_negative == 0:
            errors.append({"target_quadrant": target_quadrant, "model": "all", "selector": "all", "error": "target sem classes positiva e negativa"})
            continue
        for model_name, estimator in tqdm(models.items(), desc=target_code(target_quadrant), leave=False):
            for selector_k in selector_options:
                selector_name = "none" if selector_k is None else f"SelectKBest(k={selector_k})"
                pipeline = build_pipeline(estimator, selector_k=selector_k, n_features=X.shape[1])
                base_row = {
                    "target_quadrant": target_quadrant,
                    "target_label": target_display_name(target_quadrant),
                    "model": model_name,
                    "selector": selector_name,
                    "n_samples": len(y_binary),
                    "n_positive": n_positive,
                    "n_negative": n_negative,
                    "positive_rate": n_positive / len(y_binary),
                    "n_groups": groups.nunique(),
                    "n_features_requested": X.shape[1],
                    "n_features_effective": X.shape[1] if selector_k is None else min(selector_k, X.shape[1]),
                    "cv": f"GroupKFold({cv.get_n_splits()}) por song_id",
                    "feature_set": feature_set_name,
                }
                try:
                    fold_df, _, _ = evaluate_binary_groupkfold(pipeline, X, y_binary, groups, cv)
                    fold_metric_frames.append(fold_df.assign(model=model_name, selector=selector_name, target_quadrant=target_quadrant, target_label=target_display_name(target_quadrant)))
                    row = {**base_row, "status": "ok"}
                    row.update(summarize_binary_fold_metrics(fold_df))
                    results_rows.append(row)
                except Exception as exc:
                    row = {**base_row, "status": f"error: {type(exc).__name__}: {exc}"}
                    results_rows.append(row)
                    errors.append({"target_quadrant": target_quadrant, "model": model_name, "selector": selector_name, "error": str(exc)})
    results = sort_binary_results(pd.DataFrame(results_rows))
    fold_metrics = pd.concat(fold_metric_frames, ignore_index=True) if fold_metric_frames else pd.DataFrame()
    if errors:
        print(f"[AVISO] {len(errors)} erros durante a avaliacao binaria.")
        display(pd.DataFrame(errors).head(20))
    return results, fold_metrics


## 7. Validacao oficial - quatro modelos one-vs-rest


In [15]:
X = MODEL_SOURCE_DF[usable_feature_cols].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
quadrant_labels = MODEL_SOURCE_DF[QUADRANT_COL].astype(str)
groups = MODEL_SOURCE_DF[SONG_ID_COL].astype(str)

n_groups = groups.nunique()
cv_splits = min(N_SPLITS, n_groups)
if cv_splits < 2:
    raise RuntimeError("Sao necessarios pelo menos 2 grupos de song_id para GroupKFold.")
cv = GroupKFold(n_splits=cv_splits)

binary_class_balance = []
for target_quadrant in QUADRANT_ORDER:
    y_tmp = make_binary_target(quadrant_labels, target_quadrant)
    n_positive = int(y_tmp.sum())
    n_negative = int(len(y_tmp) - n_positive)
    binary_class_balance.append({
        "target_quadrant": target_quadrant,
        "target_label": target_display_name(target_quadrant),
        "n_positive": n_positive,
        "n_negative": n_negative,
        "positive_rate": n_positive / len(y_tmp),
    })

binary_class_balance = pd.DataFrame(binary_class_balance)
save_table(binary_class_balance, "binary_class_balance_by_quadrant.csv")
display(Markdown("### Balanco binario por quadrante alvo"))
display(binary_class_balance)

print(f"Amostras: {len(X):,} | grupos (musicas): {n_groups} | folds: {cv_splits}")
print(f"Features: {len(usable_feature_cols)}")

binary_results, binary_fold_metrics = run_binary_benchmark(
    X=X,
    quadrant_labels=quadrant_labels,
    groups=groups,
    cv=cv,
    feature_set_name=FEATURE_SET_NAME,
)

save_table(binary_results, BINARY_RESULTS_FILENAME)
if not binary_fold_metrics.empty:
    save_table(binary_fold_metrics, BINARY_FOLD_METRICS_FILENAME)

ok_binary_results = binary_results[binary_results["status"].eq("ok")].copy()
if ok_binary_results.empty:
    raise RuntimeError("Nenhum modelo binario foi avaliado com sucesso.")

best_binary_rows = select_best_per_quadrant(binary_results, exclude_dummy=True)
save_table(best_binary_rows, BINARY_BEST_FILENAME)

display(Markdown("### Melhores modelos por quadrante alvo (criterio: positive_f1_mean)"))
display(best_binary_rows[[
    "target_label", "model", "selector", "positive_f1_mean", "positive_f1_std",
    "balanced_accuracy_mean", "positive_precision_mean", "positive_recall_mean", "n_positive", "n_negative",
]])

plot_best_binary_results(best_binary_rows, "binary_best_positive_f1_by_quadrant", "Melhor F1 positivo por quadrante alvo")


# Painel multi-metrica: F1, Precision, Recall, Balanced Acc por quadrante
plot_metrics_radar_by_quadrant(
    best_binary_rows,
    "binary_multi_metric_by_quadrant",
    "Multi-Metrica: F1 / Precision / Recall / Balanced Acc por Quadrante",
)

## 7.1 Validação com split holdout 70/20/10 por song_id

Protocolo alternativo ao `GroupKFold`: divisão única das músicas em **treino (70%) / teste (20%) / validação (10%)**, mantendo a restrição anti-vazamento por `song_id`.

Para cada quadrante emocional, treina um classificador binário *one-vs-rest* no conjunto de treino (70% das músicas) e avalia no conjunto de teste (20% das músicas). O F1 da classe positiva (`positive_f1`) é a métrica principal de comparação.

In [16]:
# =========================
# SPLIT HOLDOUT 70 / 20 / 10 POR song_id — BINÁRIO POR QUADRANTE
# Divisão por músicas (não por blocos) — mantém restrição anti-vazamento
# =========================

if "X" not in dir() or "quadrant_labels" not in dir():
    print("[AVISO] Células da seção 7 não executadas — pulando holdout.")
else:
    from sklearn.base import clone as _sk_clone_b

    # ── 1. Dividir músicas ────────────────────────────────────────────────
    _all_songs_b = np.array(sorted(MODEL_SOURCE_DF[SONG_ID_COL].unique()))
    _rng_b = np.random.default_rng(RANDOM_STATE)
    _rng_b.shuffle(_all_songs_b)

    _n_total_b = len(_all_songs_b)
    _n_train_b = int(0.70 * _n_total_b)
    _n_test_b  = int(0.20 * _n_total_b)

    _songs_train_b = set(_all_songs_b[:_n_train_b])
    _songs_test_b  = set(_all_songs_b[_n_train_b : _n_train_b + _n_test_b])
    _songs_val_b   = set(_all_songs_b[_n_train_b + _n_test_b :])

    _mask_tr_b = MODEL_SOURCE_DF[SONG_ID_COL].isin(_songs_train_b)
    _mask_te_b = MODEL_SOURCE_DF[SONG_ID_COL].isin(_songs_test_b)
    _mask_va_b = MODEL_SOURCE_DF[SONG_ID_COL].isin(_songs_val_b)

    _Xtr_b = X.loc[_mask_tr_b]
    _Xte_b = X.loc[_mask_te_b]
    _Xva_b = X.loc[_mask_va_b]
    _ql_tr = quadrant_labels.loc[_mask_tr_b]
    _ql_te = quadrant_labels.loc[_mask_te_b]
    _ql_va = quadrant_labels.loc[_mask_va_b]

    print(f"Split holdout por song_id  (random_state={RANDOM_STATE})")
    print(f"  Músicas — treino: {len(_songs_train_b):>4} ({len(_songs_train_b)/_n_total_b:.1%})  "
          f"teste: {len(_songs_test_b):>4} ({len(_songs_test_b)/_n_total_b:.1%})  "
          f"val: {len(_songs_val_b):>4} ({len(_songs_val_b)/_n_total_b:.1%})")
    print(f"  Blocos  — treino: {_Xtr_b.shape[0]:>6}  teste: {_Xte_b.shape[0]:>6}  val: {_Xva_b.shape[0]:>6}")
    print()

    # ── 2. Treinar e avaliar por quadrante ────────────────────────────────
    _holdout_binary_rows = []
    _sel_opts_b = [None] + [k for k in SELECTOR_K_VALUES if k < len(usable_feature_cols)]

    for _target_q in QUADRANT_ORDER:
        _ytr_b = make_binary_target(_ql_tr, _target_q)
        _yte_b = make_binary_target(_ql_te, _target_q)
        _yva_b = make_binary_target(_ql_va, _target_q)

        _best_f1 = -1.0
        _best_row_b = None

        for _mname_b, _est_b in build_candidate_models().items():
            for _k_b in _sel_opts_b:
                try:
                    _pipe_b = build_pipeline(_sk_clone_b(_est_b), selector_k=_k_b, n_features=len(usable_feature_cols))
                    _pipe_b.fit(_Xtr_b, _ytr_b)
                    for _sp, _Xe_b, _ye_b, _ns_b in [
                        ("test",       _Xte_b, _yte_b, len(_songs_test_b)),
                        ("validation", _Xva_b, _yva_b, len(_songs_val_b)),
                    ]:
                        _yp_b = _pipe_b.predict(_Xe_b)
                        _m_b = binary_classification_metrics(_ye_b, _yp_b)
                        _row_b = {
                            "target_quadrant": _target_q,
                            "target_label":    target_display_name(_target_q),
                            "model":           _mname_b,
                            "selector":        f"SelectKBest(k={_k_b})" if _k_b else "none",
                            "split":           _sp,
                            "n_songs":         _ns_b,
                            "n_blocks":        len(_ye_b),
                            "cv":              "holdout_70_20_10",
                            "status":          "ok",
                            **_m_b,
                        }
                        _holdout_binary_rows.append(_row_b)
                        if _sp == "test" and _m_b["positive_f1"] > _best_f1:
                            _best_f1 = _m_b["positive_f1"]
                            _best_row_b = _row_b
                except Exception as _exc_b:
                    _holdout_binary_rows.append({
                        "target_quadrant": _target_q,
                        "model": _mname_b,
                        "selector": f"SelectKBest(k={_k_b})" if _k_b else "none",
                        "split": "test", "status": f"error: {_exc_b}",
                    })

        if _best_row_b:
            print(f"  {_target_q}: melhor modelo holdout → {_best_row_b['model']} / "
                  f"{_best_row_b['selector']} | F1+={_best_f1:.4f}")

    holdout_binary_df = pd.DataFrame(_holdout_binary_rows)
    save_table(holdout_binary_df, "binary_holdout_70_20_10_results.csv")

    # ── 3. Resumo — melhor modelo por quadrante no teste ─────────────────
    _hb_test_ok = holdout_binary_df[
        (holdout_binary_df["split"] == "test") & (holdout_binary_df["status"] == "ok")
    ]
    _best_holdout_per_q = (
        _hb_test_ok
        .sort_values("positive_f1", ascending=False)
        .groupby("target_quadrant", sort=False)
        .first()
        .reset_index()
    )
    save_table(_best_holdout_per_q, "binary_holdout_best_per_quadrant.csv")

    display(Markdown("### Melhores modelos por quadrante — Holdout teste (20% das músicas)"))
    display(_best_holdout_per_q[
        ["target_quadrant", "target_label", "model", "selector",
         "positive_f1", "positive_precision", "positive_recall",
         "balanced_accuracy", "n_songs", "n_blocks"]
    ].round(4))

    # ── 4. Comparação com GroupKFold ───────────────────────────────────────
    if "best_binary_rows" in dir() and not best_binary_rows.empty:
        _cmp_rows = []
        for _tq in QUADRANT_ORDER:
            _kf = best_binary_rows[best_binary_rows["target_quadrant"].astype(str) == str(_tq)]
            _ho = _best_holdout_per_q[_best_holdout_per_q["target_quadrant"].astype(str) == str(_tq)]
            if _kf.empty or _ho.empty:
                continue
            _kf_f1 = float(_kf.iloc[0].get("positive_f1_mean", float("nan")))
            _ho_f1 = float(_ho.iloc[0]["positive_f1"])
            _cmp_rows.append({
                "Quadrante":    _tq,
                "Label":        target_display_name(_tq),
                "F1+ GroupKFold(5)": round(_kf_f1, 4),
                "F1+ Holdout-test":  round(_ho_f1, 4),
                "Δ F1+":             round(_ho_f1 - _kf_f1, 4),
            })
        if _cmp_rows:
            _cmp_df = pd.DataFrame(_cmp_rows)
            save_table(_cmp_df, "binary_kfold_vs_holdout_comparison.csv")
            display(Markdown("### Comparação GroupKFold(5) vs Holdout 70/20/10 por quadrante"))
            display(_cmp_df)

    print("[Holdout binário 70/20/10] concluído.")

Split holdout por song_id  (random_state=42)
  Músicas — treino: 1261 (70.0%)  teste:  360 (20.0%)  val:  181 (10.0%)
  Blocos  — treino:   4723  teste:   1183  val:    600

  Q1_Alegre_Energetico: melhor modelo holdout → LogisticRegression_balanced / SelectKBest(k=100) | F1+=0.7744
  Q2_Tenso_Raivoso: melhor modelo holdout → LogisticRegression_balanced / SelectKBest(k=200) | F1+=0.2646
  Q3_Triste_Melancolico: melhor modelo holdout → RidgeClassifier_balanced / SelectKBest(k=200) | F1+=0.5972
  Q4_Calmo_Relaxado: melhor modelo holdout → RidgeClassifier_balanced / SelectKBest(k=200) | F1+=0.2674
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\binary_holdout_70_20_10_results.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\binary_holdout_best_per_quadrant.csv


### Melhores modelos por quadrante — Holdout teste (20% das músicas)

,target_quadrant,target_label,model,selector,positive_f1,positive_precision,positive_recall,balanced_accuracy,n_songs,n_blocks
0,Q1_Alegre_Energetico,Q1 Alegre/Energ.,LogisticRegression_balanced,SelectKBest(k=100),0.7744,0.7789,0.7699,0.7315,360,1183
1,Q3_Triste_Melancolico,Q3 Triste/Melanc.,RidgeClassifier_balanced,SelectKBest(k=200),0.5972,0.4749,0.8043,0.7919,360,1183
2,Q4_Calmo_Relaxado,Q4 Calmo/Relaxado,RidgeClassifier_balanced,SelectKBest(k=200),0.2674,0.1788,0.5299,0.6095,360,1183
3,Q2_Tenso_Raivoso,Q2 Tenso/Raivoso,LogisticRegression_balanced,SelectKBest(k=200),0.2646,0.1805,0.4959,0.6173,360,1183


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\binary_kfold_vs_holdout_comparison.csv


### Comparação GroupKFold(5) vs Holdout 70/20/10 por quadrante

,Quadrante,Label,F1+ GroupKFold(5),F1+ Holdout-test,Δ F1+
0,Q1_Alegre_Energetico,Q1 Alegre/Energ.,0.7196,0.7744,0.0548
1,Q2_Tenso_Raivoso,Q2 Tenso/Raivoso,0.3727,0.2646,-0.1081
2,Q3_Triste_Melancolico,Q3 Triste/Melanc.,0.5578,0.5972,0.0394
3,Q4_Calmo_Relaxado,Q4 Calmo/Relaxado,0.3492,0.2674,-0.0818


[Holdout binário 70/20/10] concluído.


## 8. Predicoes OOF, relatorios e matrizes de confusao


In [17]:
best_predictions, best_reports, best_confusions, best_fold_metrics, final_binary_models = generate_best_binary_artifacts(
    best_rows=best_binary_rows,
    X=X,
    source_df=MODEL_SOURCE_DF,
    quadrant_labels=quadrant_labels,
    groups=groups,
    cv=cv,
    feature_set_name=FEATURE_SET_NAME,
)

save_table(best_predictions, BINARY_OOF_FILENAME)
save_table(best_reports, BINARY_REPORT_FILENAME)
save_table(best_confusions, BINARY_CONFUSION_FILENAME)
save_table(best_fold_metrics, BINARY_BEST_FOLD_FILENAME)

if globals().get("SAVE_MODEL_ARTIFACTS", True):
    try:
        import joblib
        model_path = MODELS_PATH / BINARY_MODEL_JOBLIB if isinstance(MODELS_PATH, Path) else os.path.join(MODELS_PATH, BINARY_MODEL_JOBLIB)
        joblib.dump(final_binary_models, model_path)
        print("Modelos finais salvos:", model_path)
    except Exception as exc:
        print(f"[AVISO] Nao foi possivel salvar os modelos finais: {exc}")

display(Markdown("### Relatorios OOF dos melhores modelos"))
display(best_reports.head(20))
display(Markdown("### Matrizes de confusao OOF dos melhores modelos"))
display(best_confusions)



Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\binary_oof_predictions_best_models_fingerprints.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\binary_classification_report_best_models_fingerprints.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\binary_confusion_matrix_best_models_fingerprints.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\binary_groupkfold_best_fold_metrics_fingerprints.csv
Modelos finais salvos: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\models\best_binary_models_fingerprints.joblib


### Relatorios OOF dos melhores modelos

,target_quadrant,target_label,model,selector,label,precision,recall,f1-score,support
0,Q1_Alegre_Energetico,Q1 Alegre/Energ.,ExtraTrees_balanced,SelectKBest(k=200),resto,0.731043,0.685923,0.707765,3289.000000
1,Q1_Alegre_Energetico,Q1 Alegre/Energ.,ExtraTrees_balanced,SelectKBest(k=200),Q1 Alegre/Energ.,0.697953,0.741996,0.719301,3217.000000
2,Q1_Alegre_Energetico,Q1 Alegre/Energ.,ExtraTrees_balanced,SelectKBest(k=200),accuracy,0.713649,0.713649,0.713649,0.713649
3,Q1_Alegre_Energetico,Q1 Alegre/Energ.,ExtraTrees_balanced,SelectKBest(k=200),macro avg,0.714498,0.713959,0.713533,6506.000000
4,Q1_Alegre_Energetico,Q1 Alegre/Energ.,ExtraTrees_balanced,SelectKBest(k=200),weighted avg,0.714681,0.713649,0.713469,6506.000000
5,Q2_Tenso_Raivoso,Q2 Tenso/Raivoso,RidgeClassifier_balanced,SelectKBest(k=100),resto,0.906235,0.683434,0.779221,5487.000000
6,Q2_Tenso_Raivoso,Q2 Tenso/Raivoso,RidgeClassifier_balanced,SelectKBest(k=100),Q2 Tenso/Raivoso,0.266470,0.619235,0.372601,1019.000000
7,Q2_Tenso_Raivoso,Q2 Tenso/Raivoso,RidgeClassifier_balanced,SelectKBest(k=100),accuracy,0.673378,0.673378,0.673378,0.673378
8,Q2_Tenso_Raivoso,Q2 Tenso/Raivoso,RidgeClassifier_balanced,SelectKBest(k=100),macro avg,0.586352,0.651334,0.575911,6506.000000
9,Q2_Tenso_Raivoso,Q2 Tenso/Raivoso,RidgeClassifier_balanced,SelectKBest(k=100),weighted avg,0.806032,0.673378,0.715534,6506.000000


### Matrizes de confusao OOF dos melhores modelos

,target_quadrant,target_label,model,selector,actual,predicted,n,normalized_by_actual
0,Q1_Alegre_Energetico,Q1 Alegre/Energ.,ExtraTrees_balanced,SelectKBest(k=200),resto,resto,2256,0.685923
1,Q1_Alegre_Energetico,Q1 Alegre/Energ.,ExtraTrees_balanced,SelectKBest(k=200),resto,Q1 Alegre/Energ.,1033,0.314077
2,Q1_Alegre_Energetico,Q1 Alegre/Energ.,ExtraTrees_balanced,SelectKBest(k=200),Q1 Alegre/Energ.,resto,830,0.258004
3,Q1_Alegre_Energetico,Q1 Alegre/Energ.,ExtraTrees_balanced,SelectKBest(k=200),Q1 Alegre/Energ.,Q1 Alegre/Energ.,2387,0.741996
4,Q2_Tenso_Raivoso,Q2 Tenso/Raivoso,RidgeClassifier_balanced,SelectKBest(k=100),resto,resto,3750,0.683434
5,Q2_Tenso_Raivoso,Q2 Tenso/Raivoso,RidgeClassifier_balanced,SelectKBest(k=100),resto,Q2 Tenso/Raivoso,1737,0.316566
6,Q2_Tenso_Raivoso,Q2 Tenso/Raivoso,RidgeClassifier_balanced,SelectKBest(k=100),Q2 Tenso/Raivoso,resto,388,0.380765
7,Q2_Tenso_Raivoso,Q2 Tenso/Raivoso,RidgeClassifier_balanced,SelectKBest(k=100),Q2 Tenso/Raivoso,Q2 Tenso/Raivoso,631,0.619235
8,Q3_Triste_Melancolico,Q3 Triste/Melanc.,RidgeClassifier_balanced,SelectKBest(k=200),resto,resto,3873,0.755855
9,Q3_Triste_Melancolico,Q3 Triste/Melanc.,RidgeClassifier_balanced,SelectKBest(k=200),resto,Q3 Triste/Melanc.,1251,0.244145


In [18]:
threshold_analysis_fingerprints_by_quadrant = build_threshold_analysis(best_predictions)
save_table(threshold_analysis_fingerprints_by_quadrant, "threshold_analysis_fingerprints_by_quadrant.csv")

fingerprint_vs_opensmile_by_quadrant = build_fingerprint_vs_opensmile_comparison(best_reports)
save_table(fingerprint_vs_opensmile_by_quadrant, "fingerprint_vs_opensmile_by_quadrant.csv")

threshold_best_by_quadrant = (
    threshold_analysis_fingerprints_by_quadrant
    .sort_values(["target_quadrant", "positive_f1_mean", "balanced_accuracy_mean", "threshold"], ascending=[True, False, False, True])
    .drop_duplicates(subset=["target_quadrant"])
    .reset_index(drop=True)
    if not threshold_analysis_fingerprints_by_quadrant.empty
    else pd.DataFrame()
)

if not threshold_analysis_fingerprints_by_quadrant.empty:
    display(Markdown("### Threshold sweep por quadrante"))
    display(threshold_best_by_quadrant[[
        "target_label", "threshold", "positive_f1_mean", "positive_precision_mean",
        "positive_recall_mean", "balanced_accuracy_mean",
    ]])

if not fingerprint_vs_opensmile_by_quadrant.empty:
    display(Markdown("### Comparacao fingerprint vs openSMILE por quadrante"))
    display(fingerprint_vs_opensmile_by_quadrant[[
        "target_label", "model", "selector", "fingerprint_f1", "opensmile_f1", "f1_delta",
        "fingerprint_precision", "opensmile_precision", "precision_delta",
        "fingerprint_recall", "opensmile_recall", "recall_delta",
    ]])


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\threshold_analysis_fingerprints_by_quadrant.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\fingerprint_vs_opensmile_by_quadrant.csv


### Threshold sweep por quadrante

,target_label,threshold,positive_f1_mean,positive_precision_mean,positive_recall_mean,balanced_accuracy_mean
0,Q1 Alegre/Energ.,0.4,0.726982,0.636194,0.847995,0.686843
1,Q2 Tenso/Raivoso,0.5,0.372601,0.266470,0.619235,0.651334
2,Q3 Triste/Melanc.,0.5,0.560416,0.450351,0.741679,0.748767
3,Q4 Calmo/Relaxado,0.5,0.349564,0.233021,0.699324,0.667747


### Comparacao fingerprint vs openSMILE por quadrante

,target_label,model,selector,fingerprint_f1,opensmile_f1,f1_delta,fingerprint_precision,opensmile_precision,precision_delta,fingerprint_recall,opensmile_recall,recall_delta
1,Q1 Alegre/Energ.,ExtraTrees_balanced,SelectKBest(k=200),0.719301,NaN,NaN,0.697953,NaN,NaN,0.741996,NaN,NaN
6,Q2 Tenso/Raivoso,RidgeClassifier_balanced,SelectKBest(k=100),0.372601,NaN,NaN,0.266470,NaN,NaN,0.619235,NaN,NaN
11,Q3 Triste/Melanc.,RidgeClassifier_balanced,SelectKBest(k=200),0.560416,NaN,NaN,0.450351,NaN,NaN,0.741679,NaN,NaN
16,Q4 Calmo/Relaxado,LogisticRegression_balanced,SelectKBest(k=100),0.349564,NaN,NaN,0.233021,NaN,NaN,0.699324,NaN,NaN


## 9. Ablation por origem das fingerprints


In [19]:
_origin_map = {
    "band_rank": [c for c in usable_feature_cols if c.startswith("fpband__")],
    "rank": [c for c in usable_feature_cols if c.startswith("fprank__")],
    "fingerprint_completo": list(usable_feature_cols),
}
origin_groups = {k: v for k, v in _origin_map.items() if v}
origin_ablation = run_binary_ablation_study(origin_groups, X, quadrant_labels, groups, cv, study_name="origem", feature_set_name=FEATURE_SET_NAME)
origin_ablation_best = best_ablation_per_group(origin_ablation)
save_table(origin_ablation, "ablation_by_origin_fingerprints_by_quadrant.csv")
save_table(origin_ablation_best, "ablation_by_origin_fingerprints_by_quadrant_best.csv")
plot_ablation_summary(origin_ablation_best, "ablation_by_origin_fingerprints_by_quadrant", "Ablation por origem - F1 positivo por quadrante")



Ablation origem: 100%|██████████| 4/4 [08:00<00:00, 120.08s/it]

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\ablation_by_origin_fingerprints_by_quadrant.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\ablation_by_origin_fingerprints_by_quadrant_best.csv


## 10. Ablation por banda de frequencia


In [20]:
band_groups = {}
for band in ["low", "mid", "high", "very_high", "global"]:
    cols = [c for c in inventory.loc[inventory["banda"].eq(band), "feature"].tolist() if c in usable_feature_cols]
    if cols:
        band_groups[band] = cols
band_ablation = run_binary_ablation_study(band_groups, X, quadrant_labels, groups, cv, study_name="banda", feature_set_name=FEATURE_SET_NAME)
band_ablation_best = best_ablation_per_group(band_ablation)
save_table(band_ablation, "ablation_by_band_fingerprints_by_quadrant.csv")
save_table(band_ablation_best, "ablation_by_band_fingerprints_by_quadrant_best.csv")
plot_ablation_summary(band_ablation_best, "ablation_by_band_fingerprints_by_quadrant", "Ablation por banda - F1 positivo por quadrante")



Ablation banda:   0%|          | 0/4 [00:00<?, ?it/s]

Ablation banda: 100%|██████████| 4/4 [09:38<00:00, 144.60s/it]

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\ablation_by_band_fingerprints_by_quadrant.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\ablation_by_band_fingerprints_by_quadrant_best.csv


## 11. Ablation por tipo de feature


In [21]:
type_groups = {}
for metric_type in ["magnitude", "rank", "pitch_octave", "counts", "dispersion", "central_tendency", "frequency", "other"]:
    cols = [c for c in inventory.loc[inventory["metrica"].eq(metric_type), "feature"].tolist() if c in usable_feature_cols]
    if cols:
        type_groups[metric_type] = cols
type_ablation = run_binary_ablation_study(type_groups, X, quadrant_labels, groups, cv, study_name="tipo", feature_set_name=FEATURE_SET_NAME)
type_ablation_best = best_ablation_per_group(type_ablation)
save_table(type_ablation, "ablation_by_type_fingerprints_by_quadrant.csv")
save_table(type_ablation_best, "ablation_by_type_fingerprints_by_quadrant_best.csv")
plot_ablation_summary(type_ablation_best, "ablation_by_type_fingerprints_by_quadrant", "Ablation por tipo de feature - F1 positivo por quadrante")



Ablation tipo: 100%|██████████| 4/4 [05:28<00:00, 82.14s/it]

Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\ablation_by_type_fingerprints_by_quadrant.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\ablation_by_type_fingerprints_by_quadrant_best.csv


In [22]:
best_fingerprint_ablation_by_quadrant = build_best_fingerprint_ablation_by_quadrant(
    ("origem", origin_ablation),
    ("banda", band_ablation),
    ("metrica", type_ablation),
)
save_table(best_fingerprint_ablation_by_quadrant, "best_fingerprint_ablation_by_quadrant.csv")

if not best_fingerprint_ablation_by_quadrant.empty:
    display(Markdown("### Melhor ablation por quadrante e tipo de corte"))
    display(best_fingerprint_ablation_by_quadrant[[
        "target_label", "ablation_type", "feature_group", "model",
        "positive_f1_mean", "positive_precision_mean", "positive_recall_mean",
        "balanced_accuracy_mean", "n_features",
    ]])


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\best_fingerprint_ablation_by_quadrant.csv


## 12. Associacao das fingerprints por quadrante, tecnica e categoria de coluna

Esta etapa mede a associacao one-vs-rest entre cada feature de fingerprint e cada quadrante, agregando os resultados por banda, metrica, tecnica de coluna e origem. A leitura estatistica e cruzada com as ablations existentes quando elas ja tiverem sido executadas.


In [23]:
# === FINGERPRINT_CORRELATION_START ===
FINGERPRINT_CORRELATION_TOP_N = globals().get("FINGERPRINT_CORRELATION_TOP_N", 5)
FINGERPRINT_DIRECTION_ORDER = ["positiva", "negativa"]
FINGERPRINT_DIRECTION_COLORS = {
    "positiva": "#2563EB",
    "negativa": "#DC2626",
}
FINGERPRINT_GROUPING_LABELS = {
    "banda": "Banda",
    "metrica": "Métrica",
    "tecnica_coluna": "Técnica da coluna",
    "origem_feature": "Origem da feature",
}
FINGERPRINT_CORRELATION_AXIS_TITLE = "Média de |correlação de Pearson|"


def wrap_label(text, width=28):
    text = str(text)
    if len(text) <= width:
        return text
    words = text.replace("_", "_ ").replace("/", "/ ").split()
    lines, current = [], ""
    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) > width and current:
            lines.append(current.strip())
            current = word
        else:
            current = candidate
    if current:
        lines.append(current.strip())
    return "<br>".join(lines)


def safe_pearson_corr(x_values, y_values):
    x = pd.to_numeric(pd.Series(x_values), errors="coerce").replace([np.inf, -np.inf], np.nan)
    y = pd.to_numeric(pd.Series(y_values), errors="coerce").replace([np.inf, -np.inf], np.nan)
    mask = x.notna() & y.notna()
    if mask.sum() < 3:
        return np.nan
    x_valid = x.loc[mask]
    y_valid = y.loc[mask]
    if x_valid.nunique(dropna=True) < 2 or y_valid.nunique(dropna=True) < 2:
        return np.nan
    return float(np.corrcoef(x_valid.to_numpy(dtype=float), y_valid.to_numpy(dtype=float))[0, 1])


def correlation_sign_label(mean_signed_corr):
    if pd.isna(mean_signed_corr) or not np.isfinite(mean_signed_corr):
        return "positiva"
    return "negativa" if float(mean_signed_corr) < 0 else "positiva"


def dominant_correlation_direction(mean_signed_corr, n_positive_corr, n_negative_corr, n_valid_corr):
    return correlation_sign_label(mean_signed_corr)


def ensure_fingerprint_inventory_columns(inventory_df):
    out = inventory_df.copy()
    if "tecnica_coluna" not in out.columns:
        out["tecnica_coluna"] = out["feature"].map(fingerprint_column_technique)
    if "origem_feature" not in out.columns:
        out["origem_feature"] = out["feature"].map(fingerprint_origin)
    return out


def build_fingerprint_feature_correlations(df_in, feature_columns, inventory_df):
    inventory_local = ensure_fingerprint_inventory_columns(inventory_df)
    merge_cols = [
        "feature", "banda", "metrica", "tecnica_coluna",
        "origem_feature", "missing_rate",
    ]
    inventory_local = inventory_local[[c for c in merge_cols if c in inventory_local.columns]].copy()
    inventory_local = inventory_local[inventory_local["feature"].isin(feature_columns)].copy()

    X_corr = df_in[feature_columns].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
    rows = []
    for target_quadrant in QUADRANT_ORDER:
        y_binary = make_binary_target(df_in, target_quadrant)
        for feature in feature_columns:
            r = safe_pearson_corr(X_corr[feature], y_binary)
            rows.append({
                "target_quadrant": target_quadrant,
                "target_label": target_display_name(target_quadrant),
                "feature": feature,
                "pearson_r": r,
                "abs_pearson_r": abs(r) if np.isfinite(r) else np.nan,
            })
    feature_corr = pd.DataFrame(rows).merge(inventory_local, on="feature", how="left")
    for col in ["banda", "metrica", "tecnica_coluna", "origem_feature"]:
        if col in feature_corr.columns:
            feature_corr[col] = feature_corr[col].fillna("other")
    return feature_corr


def summarize_fingerprint_correlations(feature_corr, group_col, group_label):
    summary = (
        feature_corr
        .groupby(["target_quadrant", "target_label", group_col], as_index=False)
        .agg(
            n_usable_features=("feature", "nunique"),
            n_valid_correlations=("pearson_r", "count"),
            mean_abs_correlation=("abs_pearson_r", "mean"),
            median_abs_correlation=("abs_pearson_r", "median"),
            max_abs_correlation=("abs_pearson_r", "max"),
            mean_signed_correlation=("pearson_r", "mean"),
            n_positive_correlations=("pearson_r", lambda s: int((s > 0).sum())),
            n_negative_correlations=("pearson_r", lambda s: int((s < 0).sum())),
        )
        .rename(columns={group_col: "category_value"})
    )
    summary.insert(2, "grouping", group_label)
    summary["dominant_direction"] = summary.apply(
        lambda row: dominant_correlation_direction(
            row["mean_signed_correlation"],
            row["n_positive_correlations"],
            row["n_negative_correlations"],
            row["n_valid_correlations"],
        ),
        axis=1,
    )
    return summary.sort_values(
        ["target_quadrant", "mean_abs_correlation", "max_abs_correlation"],
        ascending=[True, False, False],
        na_position="last",
    ).reset_index(drop=True)


def validate_fingerprint_correlation_summary(summary_df, grouping, expected_values):
    missing_quadrants = sorted(set(QUADRANT_ORDER) - set(summary_df["target_quadrant"].astype(str)))
    if missing_quadrants:
        raise RuntimeError(f"Analise {grouping} sem quadrantes: {missing_quadrants}")
    missing_by_quadrant = {}
    for target_quadrant in QUADRANT_ORDER:
        present = set(summary_df.loc[summary_df["target_quadrant"].eq(target_quadrant), "category_value"])
        missing = sorted(set(expected_values) - present)
        if missing:
            missing_by_quadrant[target_quadrant] = missing
    if missing_by_quadrant:
        raise RuntimeError(f"Analise {grouping} sem categorias esperadas: {missing_by_quadrant}")


def top_fingerprint_categories(*summary_frames, top_n=FINGERPRINT_CORRELATION_TOP_N):
    frames = [df for df in summary_frames if df is not None and not df.empty]
    if not frames:
        return pd.DataFrame()
    all_summary = pd.concat(frames, ignore_index=True)
    return (
        all_summary
        .sort_values(
            ["grouping", "target_quadrant", "mean_abs_correlation", "max_abs_correlation"],
            ascending=[True, True, False, False],
            na_position="last",
        )
        .groupby(["grouping", "target_quadrant"], group_keys=False)
        .head(int(top_n))
        .reset_index(drop=True)
    )


def ablation_frame_for_comparison(ablation_name, grouping_label):
    if ablation_name not in globals():
        print(f"[AVISO] {ablation_name} nao disponivel; comparacao {grouping_label} ficara sem metricas preditivas.")
        return pd.DataFrame()
    ablation_df = globals()[ablation_name]
    if ablation_df is None or ablation_df.empty:
        print(f"[AVISO] {ablation_name} vazio; comparacao {grouping_label} ficara sem metricas preditivas.")
        return pd.DataFrame()
    cols = [
        "target_quadrant", "group", "model", "selector",
        "positive_f1_mean", "balanced_accuracy_mean",
        "positive_precision_mean", "positive_recall_mean",
    ]
    available = [c for c in cols if c in ablation_df.columns]
    out = ablation_df[available].copy().rename(columns={"group": "category_value"})
    out.insert(0, "grouping", grouping_label)
    return out


def build_fingerprint_correlation_vs_ablation(band_summary, metric_summary, technique_summary, origin_summary):
    statistical = pd.concat(
        [band_summary, metric_summary, technique_summary, origin_summary],
        ignore_index=True,
    )
    ablation_frames = [
        ablation_frame_for_comparison("band_ablation_best", "banda"),
        ablation_frame_for_comparison("type_ablation_best", "metrica"),
        ablation_frame_for_comparison("origin_ablation_best", "origem_feature"),
    ]
    ablation_frames = [df for df in ablation_frames if not df.empty]
    if not ablation_frames:
        out = statistical.copy()
        for col in ["model", "selector", "positive_f1_mean", "balanced_accuracy_mean", "positive_precision_mean", "positive_recall_mean"]:
            out[col] = np.nan
        return out
    ablation_all = pd.concat(ablation_frames, ignore_index=True)
    return statistical.merge(
        ablation_all,
        on=["grouping", "target_quadrant", "category_value"],
        how="left",
    )


def plot_fingerprint_correlation_heatmap(summary_df, filename, title):
    if summary_df.empty:
        return
    pivot = summary_df.pivot_table(
        index="category_value",
        columns="target_label",
        values="mean_abs_correlation",
        aggfunc="mean",
    )
    fig = px.imshow(
        pivot,
        text_auto=".3f",
        aspect="auto",
        color_continuous_scale="Viridis",
        title=title,
    )
    fig.update_layout(
        xaxis_title="Quadrante alvo",
        yaxis_title="Categoria",
        coloraxis_colorbar_title="|r| medio",
    )
    save_fig(fig, filename)


def plot_fingerprint_ablation_heatmap(comparison_df, grouping, filename, title):
    if "positive_f1_mean" not in comparison_df.columns:
        return
    plot_df = comparison_df[
        comparison_df["grouping"].eq(grouping)
        & comparison_df["positive_f1_mean"].notna()
    ].copy()
    if plot_df.empty:
        print(f"[AVISO] Heatmap preditivo ignorado para {grouping}: positive_f1_mean nao disponivel.")
        return
    pivot = plot_df.pivot_table(
        index="category_value",
        columns="target_label",
        values="positive_f1_mean",
        aggfunc="max",
    )
    fig = px.imshow(
        pivot,
        text_auto=".3f",
        aspect="auto",
        color_continuous_scale="Blues",
        zmin=0,
        zmax=1,
        title=title,
    )
    fig.update_layout(
        xaxis_title="Quadrante alvo",
        yaxis_title="Categoria",
        coloraxis_colorbar_title="F1 positivo",
    )
    save_fig(fig, filename)


def format_correlation_value(value, digits=3):
    if pd.isna(value):
        return ""
    return f"{float(value):.{digits}f}"


def fingerprint_grouping_label(grouping):
    grouping = str(grouping)
    return FINGERPRINT_GROUPING_LABELS.get(grouping, grouping.replace("_", " ").title())


def fingerprint_plot_slug(value):
    slug = re.sub(r"[^a-z0-9]+", "_", str(value).lower()).strip("_")
    return slug or "grupo"


def prepare_top_fingerprint_plot_data(top_df):
    required_cols = [
        "grouping", "target_quadrant", "target_label", "category_value",
        "mean_abs_correlation", "mean_signed_correlation", "dominant_direction",
        "median_abs_correlation", "max_abs_correlation",
        "n_usable_features", "n_valid_correlations",
    ]
    missing = [col for col in required_cols if col not in top_df.columns]
    if missing:
        raise ValueError(f"Colunas ausentes para o gráfico de top fingerprints: {missing}")

    plot_df = top_df.dropna(subset=["mean_abs_correlation"]).copy()
    plot_df["grouping"] = plot_df["grouping"].astype(str)
    plot_df["target_quadrant"] = plot_df["target_quadrant"].astype(str)
    plot_df["dominant_direction"] = plot_df["mean_signed_correlation"].map(correlation_sign_label)
    plot_df["grouping_label"] = plot_df["grouping"].map(fingerprint_grouping_label)
    plot_df["category_label"] = plot_df["category_value"].map(lambda x: wrap_label(x, width=18))
    plot_df["corr_text"] = plot_df["mean_abs_correlation"].map(format_correlation_value)
    plot_df["signed_text"] = plot_df["mean_signed_correlation"].map(format_correlation_value)
    plot_df["bar_color"] = plot_df["dominant_direction"].map(FINGERPRINT_DIRECTION_COLORS)
    return plot_df


def ordered_fingerprint_groupings(plot_df):
    preferred_order = ["banda", "metrica", "tecnica_coluna", "origem_feature"]
    present = list(plot_df["grouping"].drop_duplicates())
    ordered = [group for group in preferred_order if group in present]
    ordered.extend([group for group in present if group not in ordered])
    return ordered


def ordered_fingerprint_quadrants(plot_df):
    present = set(plot_df["target_quadrant"].astype(str))
    ordered = [quadrant for quadrant in QUADRANT_ORDER if str(quadrant) in present]
    ordered.extend([quadrant for quadrant in plot_df["target_quadrant"].drop_duplicates() if quadrant not in ordered])
    return ordered


def add_direction_legend(fig):
    for direction in FINGERPRINT_DIRECTION_ORDER:
        fig.add_trace(
            go.Scatter(
                x=[None], y=[None], mode="markers",
                marker=dict(symbol="square", size=12, color=FINGERPRINT_DIRECTION_COLORS[direction]),
                name=direction,
                legendgroup=direction,
                showlegend=True,
                hoverinfo="skip",
            ),
            row=1,
            col=1,
        )


def make_top_fingerprint_bar_figure(plot_df, groupings, title, height=None):
    if plot_df.empty:
        return go.Figure()

    quadrants = ordered_fingerprint_quadrants(plot_df)
    quadrant_labels = (
        plot_df.drop_duplicates("target_quadrant")
        .set_index("target_quadrant")["target_label"]
        .to_dict()
    )
    row_titles = [fingerprint_grouping_label(group) for group in groupings] if len(groupings) > 1 else None
    fig = make_subplots(
        rows=len(groupings),
        cols=len(quadrants),
        shared_xaxes=False,
        shared_yaxes=False,
        horizontal_spacing=0.06,
        vertical_spacing=0.085 if len(groupings) > 1 else 0.04,
        column_titles=[quadrant_labels.get(quadrant, str(quadrant)) for quadrant in quadrants],
        row_titles=row_titles,
    )

    for row_idx, grouping in enumerate(groupings, start=1):
        for col_idx, quadrant in enumerate(quadrants, start=1):
            subset = plot_df[
                plot_df["grouping"].eq(grouping)
                & plot_df["target_quadrant"].eq(str(quadrant))
            ].sort_values("mean_abs_correlation", ascending=True)

            if subset.empty:
                fig.add_annotation(
                    text="sem dados",
                    x=0.5,
                    y=0.5,
                    xref=f"x{'' if row_idx == 1 and col_idx == 1 else (row_idx - 1) * len(quadrants) + col_idx} domain",
                    yref=f"y{'' if row_idx == 1 and col_idx == 1 else (row_idx - 1) * len(quadrants) + col_idx} domain",
                    showarrow=False,
                    font=dict(color="#94A3B8", size=12),
                )
                fig.update_xaxes(range=[0, 1], row=row_idx, col=col_idx)
                continue

            customdata = subset[[
                "category_value", "grouping_label", "target_label", "mean_signed_correlation",
                "median_abs_correlation", "max_abs_correlation",
                "n_usable_features", "n_valid_correlations", "dominant_direction",
            ]].to_numpy()
            fig.add_trace(
                go.Bar(
                    x=subset["mean_abs_correlation"],
                    y=subset["category_label"],
                    orientation="h",
                    marker=dict(
                        color=subset["bar_color"],
                        line=dict(color="rgba(15, 23, 42, 0.16)", width=0.7),
                    ),
                    text=subset["corr_text"],
                    textposition="outside",
                    textfont=dict(color="#334155", size=12),
                    cliponaxis=False,
                    customdata=customdata,
                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        "Agrupamento: %{customdata[1]}<br>"
                        "Quadrante: %{customdata[2]}<br>"
                        "Direção: %{customdata[8]}<br>"
                        "|r| médio: %{x:.3f}<br>"
                        "r médio assinado: %{customdata[3]:.3f}<br>"
                        "Mediana |r|: %{customdata[4]:.3f}<br>"
                        "Máx. |r|: %{customdata[5]:.3f}<br>"
                        "Features úteis: %{customdata[6]}<br>"
                        "Correlação válida: %{customdata[7]}<extra></extra>"
                    ),
                    showlegend=False,
                ),
                row=row_idx,
                col=col_idx,
            )

            local_max = subset["mean_abs_correlation"].max()
            axis_max = max(0.03, float(local_max) * 1.32)
            fig.update_xaxes(range=[0, axis_max], row=row_idx, col=col_idx)

            fig.update_xaxes(
                title_text=FINGERPRINT_CORRELATION_AXIS_TITLE if row_idx == len(groupings) else "",
                tickformat=".2f",
                ticks="outside",
                showgrid=True,
                gridcolor="rgba(148, 163, 184, 0.28)",
                zeroline=True,
                zerolinecolor="rgba(15, 23, 42, 0.22)",
                row=row_idx,
                col=col_idx,
            )
            fig.update_yaxes(
                title_text="Categoria" if col_idx == 1 else "",
                showgrid=False,
                tickfont=dict(size=11),
                automargin=True,
                row=row_idx,
                col=col_idx,
            )

    add_direction_legend(fig)
    fig.update_layout(
        title=dict(text=title, x=0.02, xanchor="left"),
        barmode="overlay",
        bargap=0.28,
        width=1800,
        height=height or (330 * len(groupings) + 190),
        margin=dict(l=150, r=190, t=95, b=100),
        legend=dict(title_text="Direção dominante", traceorder="normal"),
        font=dict(family="Arial", size=12, color="#334155"),
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    fig.update_annotations(font=dict(size=13, color="#334155"))
    return fig


def plot_top_fingerprint_correlations(top_df, save_overview=True, save_by_category=True):
    if top_df.empty:
        return {}

    plot_df = prepare_top_fingerprint_plot_data(top_df)
    if plot_df.empty:
        print("[AVISO] Nenhuma correlação válida para plotar no top de fingerprints.")
        return {}

    figures = {}
    groupings = ordered_fingerprint_groupings(plot_df)
    if save_by_category:
        for grouping in groupings:
            grouping_label = fingerprint_grouping_label(grouping)
            grouping_fig = make_top_fingerprint_bar_figure(
                plot_df,
                [grouping],
                title=f"Top {FINGERPRINT_CORRELATION_TOP_N} categorias de fingerprints por quadrante - {grouping_label}",
                height=580,
            )
            save_fig(
                grouping_fig,
                f"fingerprint_top_correlations_{fingerprint_plot_slug(grouping)}_by_quadrant",
                width=1800,
                height=580,
            )
            figures[grouping] = grouping_fig

    if save_overview:
        overview = make_top_fingerprint_bar_figure(
            plot_df,
            groupings,
            title=f"Top {FINGERPRINT_CORRELATION_TOP_N} categorias de fingerprints por quadrante - escala independente por painel",
            height=330 * len(groupings) + 190,
        )
        save_fig(
            overview,
            "fingerprint_top_correlations_by_quadrant",
            width=1800,
            height=overview.layout.height,
            show=False,
        )
        figures["visao_geral"] = overview
    return figures


def make_fingerprint_category_profile_figure(plot_df, grouping, category_value):
    subset = plot_df[
        plot_df["grouping"].eq(str(grouping))
        & plot_df["category_value"].astype(str).eq(str(category_value))
    ].copy()
    if subset.empty:
        return None

    quadrant_order = ordered_fingerprint_quadrants(subset)
    subset["target_quadrant"] = pd.Categorical(
        subset["target_quadrant"],
        categories=quadrant_order,
        ordered=True,
    )
    subset = subset.sort_values("target_quadrant")
    customdata = subset[[
        "category_value", "grouping_label", "target_label", "mean_signed_correlation",
        "median_abs_correlation", "max_abs_correlation",
        "n_usable_features", "n_valid_correlations", "dominant_direction",
    ]].to_numpy()
    fig = go.Figure(
        go.Bar(
            x=subset["target_label"],
            y=subset["mean_abs_correlation"],
            marker=dict(
                color=subset["bar_color"],
                line=dict(color="rgba(15, 23, 42, 0.16)", width=0.7),
            ),
            text=subset["corr_text"],
            textposition="outside",
            textfont=dict(color="#334155", size=12),
            cliponaxis=False,
            customdata=customdata,
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "Agrupamento: %{customdata[1]}<br>"
                "Quadrante: %{customdata[2]}<br>"
                "Sinal médio: %{customdata[8]}<br>"
                "|r| médio: %{y:.3f}<br>"
                "r médio assinado: %{customdata[3]:.3f}<br>"
                "Mediana |r|: %{customdata[4]:.3f}<br>"
                "Máx. |r|: %{customdata[5]:.3f}<br>"
                "Features úteis: %{customdata[6]}<br>"
                "Correlação válida: %{customdata[7]}<extra></extra>"
            ),
        )
    )
    max_value = subset["mean_abs_correlation"].max()
    fig.update_layout(
        title=dict(
            text=f"{fingerprint_grouping_label(grouping)} - {category_value}",
            x=0.02,
            xanchor="left",
        ),
        width=980,
        height=460,
        margin=dict(l=80, r=55, t=80, b=110),
        showlegend=False,
        font=dict(family="Arial", size=12, color="#334155"),
        paper_bgcolor="white",
        plot_bgcolor="white",
        xaxis_title="Quadrante alvo",
        yaxis_title=FINGERPRINT_CORRELATION_AXIS_TITLE,
    )
    fig.update_yaxes(
        range=[0, max(0.03, float(max_value) * 1.25)],
        tickformat=".2f",
        showgrid=True,
        gridcolor="rgba(148, 163, 184, 0.28)",
        zeroline=True,
        zerolinecolor="rgba(15, 23, 42, 0.22)",
    )
    return fig


def plot_fingerprint_category_profiles(*summary_frames):
    frames = [df for df in summary_frames if df is not None and not df.empty]
    if not frames:
        return {}

    plot_df = prepare_top_fingerprint_plot_data(pd.concat(frames, ignore_index=True))
    figures = {}
    for grouping in ordered_fingerprint_groupings(plot_df):
        grouping_values = (
            plot_df.loc[plot_df["grouping"].eq(grouping), "category_value"]
            .astype(str)
            .drop_duplicates()
            .sort_values()
        )
        for category_value in grouping_values:
            fig = make_fingerprint_category_profile_figure(plot_df, grouping, category_value)
            if fig is None:
                continue
            filename = (
                "fingerprint_category_profile_"
                f"{fingerprint_plot_slug(grouping)}_"
                f"{fingerprint_plot_slug(category_value)}_by_quadrant"
            )
            save_fig(fig, filename, width=980, height=460, show=False)
            figures[(grouping, category_value)] = fig
    return figures


inventory = ensure_fingerprint_inventory_columns(inventory)
save_table(inventory, "feature_inventory_fingerprints.csv")

method_values = sorted(MODEL_SOURCE_DF["method"].dropna().astype(str).unique()) if "method" in MODEL_SOURCE_DF.columns else []
if len(method_values) <= 1:
    print(f"[INFO] Coluna method usada apenas como metadado; valores encontrados: {method_values}")
else:
    print(f"[INFO] Coluna method possui multiplos valores: {method_values}")

fingerprint_feature_correlations = build_fingerprint_feature_correlations(
    df_in=MODEL_SOURCE_DF,
    feature_columns=usable_feature_cols,
    inventory_df=inventory,
)
save_table(fingerprint_feature_correlations, "fingerprint_feature_correlations_by_quadrant.csv")

fingerprint_corr_by_band = summarize_fingerprint_correlations(fingerprint_feature_correlations, "banda", "banda")
fingerprint_corr_by_metric = summarize_fingerprint_correlations(fingerprint_feature_correlations, "metrica", "metrica")
fingerprint_corr_by_technique = summarize_fingerprint_correlations(fingerprint_feature_correlations, "tecnica_coluna", "tecnica_coluna")
fingerprint_corr_by_origin = summarize_fingerprint_correlations(fingerprint_feature_correlations, "origem_feature", "origem_feature")
fingerprint_corr_complete = summarize_fingerprint_correlations(
    fingerprint_feature_correlations.assign(origem_feature="fingerprint_completo"),
    "origem_feature",
    "origem_feature",
)
fingerprint_corr_by_origin = pd.concat([fingerprint_corr_by_origin, fingerprint_corr_complete], ignore_index=True)

validate_fingerprint_correlation_summary(fingerprint_corr_by_band, "banda", inventory["banda"].dropna().unique())
validate_fingerprint_correlation_summary(fingerprint_corr_by_metric, "metrica", inventory["metrica"].dropna().unique())
validate_fingerprint_correlation_summary(fingerprint_corr_by_technique, "tecnica_coluna", inventory["tecnica_coluna"].dropna().unique())

fingerprint_corr_top = top_fingerprint_categories(
    fingerprint_corr_by_band,
    fingerprint_corr_by_metric,
    fingerprint_corr_by_technique,
)
fingerprint_corr_vs_ablation = build_fingerprint_correlation_vs_ablation(
    fingerprint_corr_by_band,
    fingerprint_corr_by_metric,
    fingerprint_corr_by_technique,
    fingerprint_corr_by_origin,
)

save_table(fingerprint_corr_by_band, "fingerprint_correlation_by_band_quadrant.csv")
save_table(fingerprint_corr_by_metric, "fingerprint_correlation_by_metric_quadrant.csv")
save_table(fingerprint_corr_by_technique, "fingerprint_correlation_by_column_technique_quadrant.csv")
save_table(fingerprint_corr_by_origin, "fingerprint_correlation_by_origin_quadrant.csv")
save_table(fingerprint_corr_top, "fingerprint_correlation_top_by_quadrant.csv")
save_table(fingerprint_corr_vs_ablation, "fingerprint_correlation_vs_ablation_by_quadrant.csv")

plot_fingerprint_correlation_heatmap(
    fingerprint_corr_by_band,
    "fingerprint_correlation_by_band_quadrant",
    "Forca estatistica por banda de fingerprint e quadrante",
)
plot_fingerprint_correlation_heatmap(
    fingerprint_corr_by_metric,
    "fingerprint_correlation_by_metric_quadrant",
    "Forca estatistica por metrica de fingerprint e quadrante",
)
plot_fingerprint_correlation_heatmap(
    fingerprint_corr_by_technique,
    "fingerprint_correlation_by_column_technique_quadrant",
    "Forca estatistica por tecnica de coluna e quadrante",
)
plot_fingerprint_ablation_heatmap(
    fingerprint_corr_vs_ablation,
    "banda",
    "fingerprint_ablation_f1_by_band_quadrant",
    "Desempenho preditivo por banda de fingerprint e quadrante",
)
plot_fingerprint_ablation_heatmap(
    fingerprint_corr_vs_ablation,
    "metrica",
    "fingerprint_ablation_f1_by_metric_quadrant",
    "Desempenho preditivo por metrica de fingerprint e quadrante",
)
plot_top_fingerprint_correlations(fingerprint_corr_top)
plot_fingerprint_category_profiles(
    fingerprint_corr_by_band,
    fingerprint_corr_by_metric,
    fingerprint_corr_by_technique,
)

display(Markdown("### Top categorias de fingerprints por quadrante"))
display(fingerprint_corr_top[[
    "grouping", "target_label", "category_value", "mean_abs_correlation",
    "mean_signed_correlation", "dominant_direction",
    "n_usable_features", "n_valid_correlations",
]])

display(Markdown("### Fingerprints: correlacao estatistica vs ablation"))
display(fingerprint_corr_vs_ablation[[
    c for c in [
        "grouping", "target_label", "category_value", "mean_abs_correlation",
        "dominant_direction", "positive_f1_mean", "balanced_accuracy_mean",
        "model", "selector",
    ]
    if c in fingerprint_corr_vs_ablation.columns
]].head(40))
# === FINGERPRINT_CORRELATION_END ===


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\feature_inventory_fingerprints.csv
[INFO] Coluna method usada apenas como metadado; valores encontrados: ['fingerprint_band_rank']
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\fingerprint_feature_correlations_by_quadrant.csv
[AVISO] band_ablation_best vazio; comparacao banda ficara sem metricas preditivas.
[AVISO] type_ablation_best vazio; comparacao metrica ficara sem metricas preditivas.
[AVISO] origin_ablation_best vazio; comparacao origem_feature ficara sem metricas preditivas.
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\fingerprint_correlation_by_band_quadrant.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\fingerprint_correlation_by_metric_quadrant.csv
Tabela salva: C:\dev\python\TCC Ajustado\Dados

[AVISO] Heatmap preditivo ignorado para banda: positive_f1_mean nao disponivel.
[AVISO] Heatmap preditivo ignorado para metrica: positive_f1_mean nao disponivel.


### Top categorias de fingerprints por quadrante

,grouping,target_label,category_value,mean_abs_correlation,mean_signed_correlation,dominant_direction,n_usable_features,n_valid_correlations
0,banda,Q1 Alegre/Energ.,low,0.124632,0.041016,positiva,74,74
1,banda,Q1 Alegre/Energ.,very_high,0.116105,0.087153,positiva,74,74
2,banda,Q1 Alegre/Energ.,high,0.107775,0.102941,positiva,74,74
3,banda,Q1 Alegre/Energ.,global,0.085194,0.037969,positiva,213,213
4,banda,Q1 Alegre/Energ.,mid,0.056506,0.020568,positiva,74,74
5,banda,Q2 Tenso/Raivoso,high,0.092030,0.040219,positiva,74,74
6,banda,Q2 Tenso/Raivoso,very_high,0.071754,-0.002110,negativa,74,74
7,banda,Q2 Tenso/Raivoso,low,0.063359,0.041226,positiva,74,74
8,banda,Q2 Tenso/Raivoso,global,0.049161,0.036374,positiva,213,213
9,banda,Q2 Tenso/Raivoso,mid,0.044596,0.041212,positiva,74,74


### Fingerprints: correlacao estatistica vs ablation

,grouping,target_label,category_value,mean_abs_correlation,dominant_direction,positive_f1_mean,balanced_accuracy_mean,model,selector
0,banda,Q1 Alegre/Energ.,low,0.124632,positiva,NaN,NaN,NaN,NaN
1,banda,Q1 Alegre/Energ.,very_high,0.116105,positiva,NaN,NaN,NaN,NaN
2,banda,Q1 Alegre/Energ.,high,0.107775,positiva,NaN,NaN,NaN,NaN
3,banda,Q1 Alegre/Energ.,global,0.085194,positiva,NaN,NaN,NaN,NaN
4,banda,Q1 Alegre/Energ.,mid,0.056506,positiva,NaN,NaN,NaN,NaN
5,banda,Q2 Tenso/Raivoso,high,0.092030,positiva,NaN,NaN,NaN,NaN
6,banda,Q2 Tenso/Raivoso,very_high,0.071754,negativa,NaN,NaN,NaN,NaN
7,banda,Q2 Tenso/Raivoso,low,0.063359,positiva,NaN,NaN,NaN,NaN
8,banda,Q2 Tenso/Raivoso,global,0.049161,positiva,NaN,NaN,NaN,NaN
9,banda,Q2 Tenso/Raivoso,mid,0.044596,positiva,NaN,NaN,NaN,NaN


## 13. Sumario final


In [24]:
summary_rows = []
for _, row in best_binary_rows.iterrows():
    summary_rows.append({
        "Experimento": FEATURE_SET_LABEL,
        "Quadrante alvo": row["target_label"],
        "Modelo": row["model"],
        "Seletor": row["selector"],
        "F1 positivo medio": round(float(row["positive_f1_mean"]), 4),
        "F1 positivo std": round(float(row["positive_f1_std"]), 4),
        "Precision positivo medio": round(float(row["positive_precision_mean"]), 4),
        "Recall positivo medio": round(float(row["positive_recall_mean"]), 4),
        "Balanced Acc medio": round(float(row["balanced_accuracy_mean"]), 4),
        "N features": int(row["n_features_effective"]),
        "N positivo": int(row["n_positive"]),
        "N negativo": int(row["n_negative"]),
    })

summary_by_quadrant = pd.DataFrame(summary_rows)
save_table(summary_by_quadrant, "summary_best_binary_models_by_quadrant.csv")
display(Markdown("## Sumario final - modelos binarios por quadrante"))
display(summary_by_quadrant)

if not summary_by_quadrant.empty:
    fig = px.bar(
        summary_by_quadrant,
        x="Quadrante alvo",
        y="F1 positivo medio",
        color="Quadrante alvo",
        error_y="F1 positivo std",
        text=summary_by_quadrant["F1 positivo medio"].map(lambda x: f"{x:.3f}"),
        hover_data={
            "Modelo": True,
            "Seletor": True,
            "Balanced Acc medio": ":.3f",
            "Precision positivo medio": ":.3f",
            "Recall positivo medio": ":.3f",
            "N features": True,
        },
        color_discrete_map=QUADRANT_COLOR_MAP,
        title=f"Resumo final — {FEATURE_SET_LABEL}: F1 positivo por quadrante",
        labels={"F1 positivo medio": "F1 positivo médio", "Quadrante alvo": "Quadrante"},
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(
        showlegend=False,
        yaxis=dict(range=[0, 1.1]),
        margin=dict(l=10, r=10, t=60, b=10),
    )
    save_fig(fig, "summary_best_binary_models_by_quadrant")


In [25]:
fingerprint_final_by_quadrant_summary = build_fingerprint_final_summary(best_binary_rows)
save_table(fingerprint_final_by_quadrant_summary, "fingerprint_final_by_quadrant_summary.csv")

if not fingerprint_final_by_quadrant_summary.empty:
    display(Markdown("### Sumario final do fingerprint por quadrante"))
    display(fingerprint_final_by_quadrant_summary)


Tabela salva: C:\dev\python\TCC Ajustado\Dados\pycaret_quadrants_fingerprints_by_quadrant_outputs\tables\fingerprint_final_by_quadrant_summary.csv


### Sumario final do fingerprint por quadrante

,target_quadrant,target_label,model,selector,positive_f1_mean,positive_f1_std,positive_precision_mean,positive_recall_mean,balanced_accuracy_mean,n_features,n_positive,n_negative
0,Q1_Alegre_Energetico,Q1 Alegre/Energ.,ExtraTrees_balanced,SelectKBest(k=200),0.719621,0.022563,0.699847,0.743503,0.714849,200,3217,3289
1,Q2_Tenso_Raivoso,Q2 Tenso/Raivoso,RidgeClassifier_balanced,SelectKBest(k=100),0.372702,0.006243,0.266859,0.622693,0.653019,100,1019,5487
2,Q3_Triste_Melancolico,Q3 Triste/Melanc.,RidgeClassifier_balanced,SelectKBest(k=200),0.557758,0.037365,0.448500,0.743447,0.749569,200,1382,5124
3,Q4_Calmo_Relaxado,Q4 Calmo/Relaxado,LogisticRegression_balanced,SelectKBest(k=100),0.349217,0.052003,0.234144,0.698714,0.667544,100,888,5618
